# LSST DP1 asteroid pipeline — reduce, period-search, gate, BIC/FAP, SBDB

Processes **all staged Rubin/LSST (SBN X05) objects** end-to-end, in parallel
across CPU cores. Per object it:

1. **Reduces** the raw photometry — removes distance, light-time and phase-angle
   geometry (JPL HORIZONS + IAU *H,G*) to get reduced absolute magnitudes `H_reduced`.
2. **Searches for the rotation period** — multiband Lomb-Scargle, Phase Dispersion
   Minimization, and a high-order Fourier search.
3. **Cross-checks agreement** — where all three methods match (within 1%, PDM
   harmonics allowed), applies quality **gates** and an inline **BIC / permutation FAP**.
4. **Enriches** each row from the **JPL Small-Body Database** (orbit class, NEO/PHA,
   known rotation period).

Outputs: `DP1_master_summary.csv` (one row per object) and `DP1_combined_agreement.csv`
(matched periods + gates + FAP). Run top to bottom — the file-split/staging is
one-time, then the batch cell fans the machinery below out across workers.

## Part 1 &mdash; Photometric reduction

The reduction machinery, called per object by the batch driver: query JPL HORIZONS
for the observing geometry, convert times, and apply the distance / light-time /
phase-angle corrections (IAU *H,G*) to get reduced absolute magnitudes `H_reduced`.

**Libraries**: [`astroquery.jplhorizons`](https://astroquery.readthedocs.io/) queries
NASA/JPL HORIZONS for geometry, `astropy.time` does the JD&nbsp;&harr;&nbsp;UTC
conversion, `numpy` the arithmetic.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 reduce_mette.py  --  Photometric reduction of asteroid (1727) Mette
================================================================================

PURPOSE
-------
Take the raw, apparent R-band magnitudes of asteroid (1727) Mette (supplied in
ALCDEF format in "Complete Mette.txt") and turn them into physically meaningful
"reduced" magnitudes by removing the three geometry-dependent effects that make
an asteroid look brighter or fainter from night to night:

  1. DISTANCE   -- the asteroid's brightness falls off with both its distance
                   from the Sun (r) and from the observer (delta).
  2. LIGHT TIME -- the photons we record left the asteroid ~r_delta minutes
                   earlier; the timestamps are shifted back so the light curve
                   is expressed in the asteroid's own ("astrometric") time.
  3. PHASE ANGLE-- the Sun-asteroid-observer angle (alpha) changes how much of
                   the sunlit surface we see; corrected with the IAU H,G phase
                   function using the standard slope parameter G = 0.15.

The geometry (r, delta, alpha, light-time) for every observation timestamp is
obtained directly from NASA/JPL's HORIZONS ephemeris service.

METHOD / EQUATIONS
------------------
For each observation with apparent magnitude m at Julian Date JD:

  * Geometry from HORIZONS at that epoch:
        r        = Sun -> asteroid distance               [AU]
        delta    = observer -> asteroid distance          [AU]
        alpha    = phase angle (Sun-asteroid-observer)    [deg]
        lt       = one-way light time                     [min]

  * Light-time correction (applied to the TIME):
        JD_ltc = JD - lt/1440                              [days]

  * Distance correction (reduce to r = delta = 1 AU):
        m_dist = m - 5*log10(r * delta)

  * IAU H,G phase function (Bowell et al. 1989), with t = tan(alpha/2):
        Phi1(alpha) = exp(-3.33 * t**0.63)
        Phi2(alpha) = exp(-1.87 * t**1.22)
        phase_term  = 2.5*log10( (1-G)*Phi1 + G*Phi2 )     [<= 0]

  * Fully reduced (absolute) magnitude H, i.e. brightness at r=delta=1 AU and
    alpha = 0 deg:
        H = m - 5*log10(r*delta) + phase_term
          = m_dist + phase_term

    (At alpha = 0, Phi1 = Phi2 = 1 so phase_term = 0 and H = m_dist, as it must.)

The per-point photometric error is unchanged by these additive geometric
corrections, so it is carried through unmodified.

VALIDATION ("scientific and tested")
------------------------------------
  * Unit tests on the phase function (run_self_tests):
        - Phi1(0) = Phi2(0) = 1 exactly.
        - phase_term(alpha=0) = 0  -> H == m_dist at zero phase.
        - phase functions decrease monotonically with alpha.
        - a hand-checked reference value at alpha = 22.586 deg.
  * Cross-check against the data file itself: every ALCDEF session header lists
    a PHASE value; the script compares it to the mean HORIZONS alpha for that
    session and reports the deviation (should be < ~0.2 deg).

LIBRARIES
---------
  * astroquery.jplhorizons -- the standard Python interface to JPL HORIZONS;
                              builds and submits the query for us.
  * astropy.time           -- rigorous JD (UTC) <-> calendar-UTC conversion.
  * numpy                  -- vectorised arithmetic.

USAGE
-----
    python reduce_mette.py            # full reduction -> mette_reduced.csv
    python reduce_mette.py --selftest # run only the offline unit tests

HORIZONS results are cached to disk (mette_horizons_cache.csv) so re-runs do not
re-query the service.
================================================================================
"""

import os
import sys
import math
import time
import warnings

import numpy as np
from astropy.time import Time
from astroquery.jplhorizons import Horizons

warnings.simplefilter("ignore")          # keep the console output clean




### Configuration &mdash; reduction constants

Shared constants used across the pipeline: the working directory `BASE_DIR`, the
assumed slope parameter **G&nbsp;=&nbsp;0.15**, the IAU *H,G* phase-function
constants (Bowell et&nbsp;al. 1989), and the HORIZONS request batch size. (Per-object
target and output paths are set inside the batch, not here.)

In [ ]:
# ------------------------------------------------------------------------------
# CONFIGURATION  --  reduction constants used across the whole pipeline
# ------------------------------------------------------------------------------
BASE_DIR = r"E:\CLAUDE"           # working dir (input files + all per-object outputs)

# Phase-function slope parameter (assumed).
G_SLOPE = 0.15

# IAU H,G phase-function constants (Bowell et al. 1989, "Asteroids II").
A1, B1 = 3.33, 0.63
A2, B2 = 1.87, 1.22

# How many epochs to send to HORIZONS per request (keeps the URL a sane length).
HORIZONS_CHUNK = 50

### Step 2 &mdash; Time conversion (JD&nbsp;UTC &rarr; calendar UTC)

HORIZONS interprets a bare Julian Date as UTC, so we treat the file's JDs as
JD(UTC) and convert them to ISO-8601 strings with `astropy.time`.

In [3]:
# ------------------------------------------------------------------------------
# 2. TIME CONVERSION  (JD in UTC  ->  calendar UTC)
# ------------------------------------------------------------------------------
def jd_to_utc_iso(jd_array):
    """
    Convert an array of Julian Dates (UTC) to ISO-8601 UTC strings.

    HORIZONS interprets bare Julian Dates as UTC, so we treat the file's JDs as
    JD(UTC) and use astropy.time for a rigorous, leap-second-aware conversion.
    """
    t = Time(np.asarray(jd_array, dtype=float), format="jd", scale="utc")
    return t.isot                      # e.g. '2026-03-24T23:50:32.006'




### Step 3 &mdash; Query JPL HORIZONS (with an on-disk cache)

For every epoch we need the Sun&ndash;asteroid distance *r*, the
observer&ndash;asteroid distance *&delta;*, the phase angle *&alpha;* and the
one-way light time. These come straight from HORIZONS, requested in batches and
**cached to `mette_horizons_cache.csv`** so re-runs (and this notebook) don't
re-query the service.

In [ ]:
# ------------------------------------------------------------------------------
# 3. QUERY JPL HORIZONS  (with an on-disk cache)
# ------------------------------------------------------------------------------
def _load_cache(path):
    """Load the JD -> geometry cache, if present, into a dict keyed by rounded JD."""
    cache = {}
    if not os.path.isfile(path):
        return cache
    with open(path, "r", encoding="utf-8") as fh:
        header = fh.readline()                          # skip column header
        for line in fh:
            line = line.strip()
            if not line:
                continue
            jd, r, delta, alpha, lt = line.split(",")
            cache[round(float(jd), 6)] = (
                float(r), float(delta), float(alpha), float(lt)
            )
    return cache


def _save_cache(path, cache):
    """Persist the geometry cache (atomic-ish overwrite)."""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("jd,r_au,delta_au,alpha_deg,lighttime_min\n")
        for jd in sorted(cache):
            r, delta, alpha, lt = cache[jd]
            fh.write(f"{jd:.6f},{r:.10f},{delta:.10f},{alpha:.6f},{lt:.8f}\n")


def query_horizons(jds, target=None, target_type="smallbody", site=None,
                   chunk=HORIZONS_CHUNK, cache_path=None):
    """
    Fetch r, delta, alpha and one-way light time for every JD from HORIZONS.

    Uses an on-disk cache so repeated runs (and crash recovery) do not re-query
    epochs already retrieved.  Epochs are requested in batches; each returned
    row is matched back to the requested JD by nearest datetime_jd (HORIZONS
    echoes the requested epoch to ~microsecond precision).

    Returns
    -------
    dict : round(jd,6) -> (r_au, delta_au, alpha_deg, lighttime_min)
    """
    cache = _load_cache(cache_path)
    todo = [jd for jd in jds if round(jd, 6) not in cache]
    print(f"  HORIZONS: {len(jds)} epochs requested, "
          f"{len(jds) - len(todo)} cached, {len(todo)} to fetch.")

    for start in range(0, len(todo), chunk):
        batch = todo[start:start + chunk]
        # epochs as a plain list of JD floats -> HORIZONS TLIST (interpreted UTC)
        obj = Horizons(id=target, id_type=target_type, location=site,
                       epochs=list(batch))
        eph = None
        for attempt in range(3):                        # simple retry on hiccups
            try:
                eph = obj.ephemerides()    # default set includes r, delta, alpha, lighttime
                break
            except Exception as exc:
                if attempt == 2:
                    raise
                print(f"    retry {attempt+1} after error: {exc}")
                time.sleep(3)

        ret_jd = np.asarray(eph["datetime_jd"], dtype=float)
        for jd in batch:
            k = int(np.argmin(np.abs(ret_jd - jd)))
            if abs(ret_jd[k] - jd) > 1e-4:              # > ~9 s mismatch = problem
                raise RuntimeError(f"HORIZONS epoch mismatch for JD {jd}")
            cache[round(jd, 6)] = (
                float(eph["r"][k]),
                float(eph["delta"][k]),
                float(eph["alpha"][k]),
                float(eph["lighttime"][k]),
            )
        _save_cache(cache_path, cache)                  # checkpoint after each batch
        print(f"    fetched {min(start + chunk, len(todo))}/{len(todo)}")

    return cache




### Step 4 &mdash; The IAU *H,G* phase function

The phase term \(2.5\log_{10}[(1-G)\,\Phi_1 + G\,\Phi_2]\) is the
magnitude correction that reduces a measurement to zero phase angle. It is
&le;&nbsp;0 everywhere and exactly 0 at &alpha;&nbsp;=&nbsp;0.

In [5]:
# ------------------------------------------------------------------------------
# 4. IAU H,G PHASE FUNCTION
# ------------------------------------------------------------------------------
def hg_phase_term(alpha_deg, G=G_SLOPE):
    """
    Return the H,G phase term  2.5*log10[(1-G)*Phi1 + G*Phi2]  in magnitudes.

    This is the quantity ADDED to the distance-corrected magnitude to reduce it
    to zero phase angle.  It is <= 0 (an object is brightest at alpha = 0), and
    equals 0 exactly at alpha = 0.

    Works on scalars or numpy arrays.
    """
    alpha = np.radians(np.asarray(alpha_deg, dtype=float))
    t = np.tan(alpha / 2.0)
    phi1 = np.exp(-A1 * np.power(t, B1))
    phi2 = np.exp(-A2 * np.power(t, B2))
    return 2.5 * np.log10((1.0 - G) * phi1 + G * phi2)




### Step 5 &mdash; Offline unit tests

Good scientific code checks its own math *before* touching real data. These
network-free tests confirm the phase function is 1 at zero phase, decreases
monotonically with phase angle, matches a hand-computed reference value, and
that the full reduction collapses to \(H = m - 5\log_{10}(r\,\delta)\) at
&alpha;&nbsp;=&nbsp;0.

In [6]:
# ------------------------------------------------------------------------------
# 5. OFFLINE UNIT TESTS (no network needed)
# ------------------------------------------------------------------------------
def run_self_tests():
    """Validate the photometry maths before touching real data."""
    print("[selftest] running offline unit tests ...")

    # (a) phase functions are exactly 1 at zero phase -> phase term is 0
    assert abs(hg_phase_term(0.0)) < 1e-12, "phase term must be 0 at alpha=0"

    # (b) monotonic: brightness contribution decreases as phase angle grows,
    #     so the (negative) phase term becomes more negative with alpha.
    grid = np.linspace(0, 90, 19)
    terms = hg_phase_term(grid)
    assert np.all(np.diff(terms) < 0), "phase term must decrease with alpha"

    # (c) hand-checked reference value at alpha = 22.586 deg, G = 0.15.
    #     t = tan(11.293 deg) = 0.19956
    #     Phi1 = exp(-3.33 * 0.19956**0.63) = 0.29918
    #     Phi2 = exp(-1.87 * 0.19956**1.22) = 0.76982
    #     term = 2.5*log10(0.85*0.29918 + 0.15*0.76982) = -1.0802 mag
    ref = hg_phase_term(22.586, G=0.15)
    assert abs(ref - (-1.0802)) < 2e-3, f"reference phase term off: {ref:.4f}"

    # (d) full reduction identity at zero phase: H == m - 5log10(r*delta)
    m, r, delta = 15.0, 1.9, 1.1
    m_dist = m - 5.0 * math.log10(r * delta)
    H0 = m_dist + hg_phase_term(0.0)
    assert abs(H0 - m_dist) < 1e-12, "H must equal m_dist at alpha=0"

    print("[selftest] all unit tests PASSED.")




### Step 6 &mdash; Apply corrections & write output

Now we attach the geometry to each record, apply the light-time, distance and
phase corrections to get the reduced absolute magnitude **H_reduced**, write the
CSV.

In [7]:
# ------------------------------------------------------------------------------
# 6. APPLY CORRECTIONS + WRITE OUTPUT
# ------------------------------------------------------------------------------
def reduce_records(records, geom):
    """
    Attach geometry and the three corrections to every observation record.

    Adds the keys: utc, r, delta, alpha, lighttime_min, jd_ltc, m_dist,
    phase_term, H.
    """
    jds = np.array([rec["jd"] for rec in records])
    utc = jd_to_utc_iso(jds)

    for rec, u in zip(records, utc):
        r, delta, alpha, lt = geom[round(rec["jd"], 6)]
        rec["utc"] = u
        rec["r"] = r
        rec["delta"] = delta
        rec["alpha"] = alpha
        rec["lighttime_min"] = lt

        # light-time correction (to the timestamp)
        rec["jd_ltc"] = rec["jd"] - lt / 1440.0

        # distance correction (to the magnitude)
        rec["m_dist"] = rec["mag"] - 5.0 * math.log10(r * delta)

        # phase-angle correction (to the magnitude) -> fully reduced mag H
        rec["phase_term"] = float(hg_phase_term(alpha))
        rec["H"] = rec["m_dist"] + rec["phase_term"]

    return records


def write_csv(records, path):
    """Write the reduced photometry to a CSV file."""
    cols = ["session", "session_date", "filter", "jd", "utc", "jd_ltc",
            "mag", "magerr", "r_au", "delta_au", "alpha_deg",
            "lighttime_min", "m_dist", "phase_term", "H_reduced", "phase_meta"]
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(",".join(cols) + "\n")
        for rec in records:
            fh.write(",".join(str(x) for x in [
                rec["session"], rec["session_date"],
                "" if rec["filter"] is None else rec["filter"],
                f"{rec['jd']:.6f}", rec["utc"], f"{rec['jd_ltc']:.6f}",
                f"{rec['mag']:.3f}", f"{rec['magerr']:.3f}",
                f"{rec['r']:.8f}", f"{rec['delta']:.8f}", f"{rec['alpha']:.4f}",
                f"{rec['lighttime_min']:.6f}",
                f"{rec['m_dist']:.4f}", f"{rec['phase_term']:.4f}", f"{rec['H']:.4f}",
                "" if rec["phase_meta"] is None else f"{rec['phase_meta']:.2f}",
            ]) + "\n")


---

## Part 2 &mdash; Rotation-period search

With the geometry removed, the only thing varying in `H_reduced` is the rotational
light curve. Per object, three methods search for the period:

1. **Remove the obvious outliers** (robust median/MAD clip).
2. **Multiband Lomb-Scargle** (astropy) &mdash; models the per-band colour offsets.
3. **Phase Dispersion Minimization** (PyAstronomy) &mdash; model-free.
4. **Fourier** &mdash; high-order weighted-dispersion search.

> **Backend.** `matplotlib.use("Agg")` &mdash; figures are written to PNG per object.

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 find_period.py  --  Rotation period of asteroid (1727) Mette
================================================================================

PURPOSE
-------
Read the reduced photometry produced by ``reduce_mette.py``
(``mette_reduced.csv``) and search for the asteroid's rotational period by
fitting a Fourier series at many trial periods, then visualise the result.

Because the magnitudes have already been corrected for distance, light-time and
phase angle (column ``H_reduced``), the night-to-night brightness trends are
removed and the only variation left is the rotational light curve -- exactly
what a period search needs.

WHAT IT DOES
------------
  1. Loads the reduced light curve (time = light-time-corrected JD ``jd_ltc``,
     brightness = ``H_reduced``, weight = 1/``magerr``).
  2. Removes ONLY the really obvious outliers with a conservative robust clip
     (median +/- K*MAD, K=5) BEFORE any period search.
  3. Runs a FOURIER period search over trial periods 2-10 h (sampled with
     ~10,000 frequency steps): at every trial period the folded curve is fitted
     with a 4th-order Fourier series by weighted least squares, and the period
     that explains the most variance (lowest chi-square) wins.
  4. Makes three plots:
        (1) mette_lightcurve_all.png  -- H_reduced vs time, all nights.
        (2) mette_periodogram.png     -- Fourier "variance explained" vs period.
        (3) mette_folded.png          -- light curve folded at the best period,
                                         with a 4th-order Fourier fit overlaid.

A NOTE ON ASTEROID PERIODS
--------------------------
Asteroid light curves are usually double-peaked (two maxima + two minima per
rotation, from an elongated shape), so the dominant peak often corresponds to
HALF the true rotation period.  This script reports the raw dominant period (as
requested) and ALSO prints twice that value, which is the more likely physical
rotation period; inspect the folded plot to judge.

LIBRARIES
---------
  * numpy       -- weighted least-squares Fourier fit and the period search.
  * matplotlib  -- plotting.
================================================================================
"""

import os
import csv

import numpy as np
import matplotlib
matplotlib.use("Agg")                      # headless: render straight to PNG files
import matplotlib.pyplot as plt
plt.rcParams.update({            # global: larger fonts + default figure on every plot
    "font.size": 16,
    "axes.titlesize": 19,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 15,
    "legend.title_fontsize": 16,
})




### Configuration &mdash; search window & clip

The period-search window and frequency grid are set **per object** in
`analyze_rubin` (&approx;&nbsp;0.0156&nbsp;h&nbsp;&hellip;&nbsp;baseline/2, capped
72&nbsp;h; one grid shared by LS / PDM / Fourier). The Fourier order is chosen
adaptively (**k&nbsp;=&nbsp;2&ndash;4**). Obvious outliers are clipped at
**5&nbsp;robust&nbsp;&sigma;** before any search.

In [ ]:
# ------------------------------------------------------------------------------
# CONFIGURATION -- period-search window + outlier clip (shared by the batch)
# ------------------------------------------------------------------------------
# Period search window. analyze_rubin overrides these per object (0.0156 h .. baseline/2).
P_MIN_HOURS = 0.1
P_MAX_HOURS = 25

# Outlier clip: drop points > K robust sigmas (1.4826*MAD) from the median H_reduced.
# K=5 keeps the real ~0.1 mag light-curve variation, removes only obvious bad points.
CLIP_K = 5.0

HOURS_PER_DAY = 24.0

### Step 1 &mdash; Load the reduced light curve

Read the CSV columns we need: light-time-corrected time `jd_ltc`, brightness
`H_reduced`, and the photometric error (which becomes the fit weight 1/err).

In [10]:
# ------------------------------------------------------------------------------
# 1. LOAD THE REDUCED LIGHT CURVE
# ------------------------------------------------------------------------------
def load_lightcurve(path):
    """
    Read the reduced-photometry CSV.

    Returns
    -------
    dict of numpy arrays: t (days, light-time corrected JD), mag (H_reduced),
    err (magerr), session (int), and a human-readable session_date list.
    """
    t, mag, err, session, date, band = [], [], [], [], [], []
    with open(path, "r", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            t.append(float(row["jd_ltc"]))
            mag.append(float(row["H_reduced"]))
            # guard against missing/zero errors so the weighting stays finite
            try:
                e = float(row["magerr"])
            except (ValueError, KeyError):
                e = np.nan
            err.append(e)
            session.append(int(row["session"]))
            date.append(row["session_date"])
            band.append(_norm_band(row.get("filter", "")))

    t = np.array(t)
    mag = np.array(mag)
    err = np.array(err)
    session = np.array(session)

    # Replace non-finite or zero errors with the median error (robust default).
    good = np.isfinite(err) & (err > 0)
    med_err = np.median(err[good]) if good.any() else 0.02
    err[~good] = med_err

    order = np.argsort(t)                  # keep everything time-ordered
    return {
        "t": t[order], "mag": mag[order], "err": err[order],
        "session": session[order], "date": [date[i] for i in order],
        "band": [band[i] for i in order],
    }




### Step 2 &mdash; Reject only the obvious outliers

A robust **median/MAD** clip. Because the median and MAD are themselves immune
to outliers, a generous 5&sigma; cut removes only the genuinely bad points while
keeping the real &sim;0.1&nbsp;mag light-curve variation.

In [11]:
# ------------------------------------------------------------------------------
# 2. OUTLIER REJECTION (only the really obvious points)
# ------------------------------------------------------------------------------
def clip_outliers(lc, k=CLIP_K):
    """
    Robust median/MAD clip of H_reduced.  Returns (clean_lc, removed_lc).

    Uses the median and the median absolute deviation (MAD), which are immune
    to the very outliers we are trying to find.  Only points more than k robust
    sigmas from the median are dropped, so genuine light-curve variation stays.
    """
    mag = lc["mag"]
    med = np.median(mag)
    mad = np.median(np.abs(mag - med))
    sigma = 1.4826 * mad if mad > 0 else np.std(mag)
    keep = np.abs(mag - med) <= k * sigma

    def subset(mask):
        return {
            "t": lc["t"][mask], "mag": lc["mag"][mask], "err": lc["err"][mask],
            "session": lc["session"][mask],
            "date": [d for d, m in zip(lc["date"], mask) if m],
            "band": [b for b, m in zip(lc["band"], mask) if m],
        }

    return subset(keep), subset(~keep), (med, sigma)




### Step 3 &mdash; Fourier machinery

The building blocks of the search: the Fourier **design matrix**
\([1,\cos2\pi\varphi,\sin2\pi\varphi,\dots]\), a weighted least-squares
solve that returns the fit's \(\chi^2\) (via the normal equations, so it is
fast enough to call 10,000 times), and the phase-folding helper.

In [12]:
# ------------------------------------------------------------------------------
# 3. FOURIER MACHINERY  (design matrix + weighted least squares)
# ------------------------------------------------------------------------------
def fourier_design(phase, order):
    """Design matrix [1, cos2piÏ†, sin2piÏ†, cos4piÏ†, ...] for a Fourier series."""
    cols = [np.ones_like(phase)]
    for k in range(1, order + 1):
        cols.append(np.cos(2 * np.pi * k * phase))
        cols.append(np.sin(2 * np.pi * k * phase))
    return np.vstack(cols).T


def weighted_fourier_chi2(phase, mag, err, order):
    """
    Fit a Fourier series of given order by weighted least squares and return the
    weighted residual sum   chi2 = sum_i p_i (O_i - C_i)^2 ,  p_i = 1/err_i^2.

    Uses np.linalg.lstsq on the weighted design matrix (more numerically stable
    than forming/solving the normal equations).
    """
    A = fourier_design(phase, order)
    w = 1.0 / err                       # row weights: minimise sum((resid/err)^2)
    Aw = A * w[:, None]
    yw = mag * w
    coeffs, *_ = np.linalg.lstsq(Aw, yw, rcond=None)
    resid_w = Aw @ coeffs - yw
    return float(resid_w @ resid_w)


def fold_phase(t, period_days, t0):
    """Return rotational phase in [0,1) for each time."""
    return np.mod((t - t0) / period_days, 1.0)




### Step 4 &mdash; The Fourier period search (weighted dispersion &sigma;)

For every trial period we fold the light curve, fit an `order`-harmonic Fourier
series by weighted least squares (`np.linalg.lstsq`), and evaluate the paper's
**weighted, degrees-of-freedom-corrected dispersion**

$$\sigma^2 = \frac{N}{N-n_\mathrm{par}}\,
             \frac{\sum_i p_i (O_i-C_i)^2}{\sum_j p_j},
   \qquad p_i = 1/\mathrm{rms}_i^2 .$$

The best period **minimises &sigma;Â²**. `run_fourier_search` returns &sigma;Â²
over the grid; `fit_fourier` returns the fitted model.

In [13]:
# ------------------------------------------------------------------------------
# 4. FOURIER PERIOD SEARCH
# ------------------------------------------------------------------------------
def run_fourier_search(t, mag, err, p_min_h, p_max_h, n_steps, order):
    """
    Fourier period search using the paper's weighted, degrees-of-freedom
    corrected dispersion as the objective:

        sigma^2(P) = [ N / (N - n_par) ] * sum_i p_i r_i^2 / sum_j p_j ,
                     p_i = 1/err_i^2 ,   r_i = O_i - C_i ,   n_par = 2*order + 1

    This is chi^2 scaled by a data-only constant and the degrees of freedom.
    The BEST period MINIMISES sigma^2 (same argmin as chi^2 at fixed order).

    Returns
    -------
    periods_h, sigma2, best_period_h, best_sigma2
    """
    f_min = 1.0 / (p_max_h / HOURS_PER_DAY)        # cycles per day
    f_max = 1.0 / (p_min_h / HOURS_PER_DAY)
    frequency = np.linspace(f_min, f_max, n_steps)
    periods_days = 1.0 / frequency
    t0 = t.min()

    n = len(t)
    Sp = np.sum(1.0 / err ** 2)                    # sum of weights p_i
    n_par = 2 * order + 1
    scale = (n / Sp) / (n - n_par)                 # N/(sum p) / (N - n_par)

    sigma2 = np.empty(n_steps)
    for i, p_days in enumerate(periods_days):
        phase = fold_phase(t, p_days, t0)
        chi2 = weighted_fourier_chi2(phase, mag, err, order)   # sum_i p_i r_i^2
        sigma2[i] = scale * chi2

    periods_h = periods_days * HOURS_PER_DAY
    k = int(np.argmin(sigma2))
    return periods_h, sigma2, periods_h[k], sigma2[k]


def fit_fourier(phase, mag, err, order):
    """
    Weighted least-squares fit of a Fourier series to the folded light curve,
    returning the coefficients and a callable model(phase).
    """
    A = fourier_design(phase, order)
    w = 1.0 / err
    coeffs, *_ = np.linalg.lstsq(A * w[:, None], mag * w, rcond=None)

    def model(phi):
        return fourier_design(phi, order) @ coeffs

    return coeffs, model




---

# Part 3 &mdash; The pipeline as functions

Every analysis block below is a function. The batch driver (Part&nbsp;4) ships these
to worker processes and applies them to every staged object.

In [ ]:
# ==========================================================================
#  CORE HELPERS  (shared by all methods; lifted out of the analysis blocks)
# ==========================================================================
from IPython.display import Image, display
from scipy.signal import find_peaks

PEAK_PROM_FRAC = 0.10   # a maximum counts only if its prominence > this * (model peak-to-peak)
AMP_MAX_MAG = 2         # reject a candidate/fold whose model amplitude exceeds this (mag) -- all methods
MIN_POINTS  = 60        # objects with fewer raw points than this (before geometry) are skipped -- no CSV row

# --- gate thresholds: evaluated at the MATCHED period(s) for the combined CSV ------
# (see build_agreement_rows). Each becomes a yes/no column; per-method summary
# columns are NOT blanked anymore -- the models all run and record their top 1/2/3.
PHASE_COV_NBINS       = 20    # split the folded phase [0,1) into this many equal bins
PHASE_COV_MIN_PER_BIN = 2     # a bin "passes" if it holds at least this many points
PHASE_COV_MIN_BINS    = 16    # >= this many passing bins (>= 80% of 20) -> coverage "yes"
LS_POWER_MIN      = 0.3       # multiband-LS : peak (raw) power >= this -> "yes"
PDM_THETA_MAX     = 0.7       # PDM          : coverage-penalised theta <  this -> "yes"
FOURIER_SIGMA_MAX = 0.2       # Fourier      : candidate sigma <  this -> "yes"
OBS_MIN_PER_FILTER = 30       # obs/filter gate: a filter "counts" with >= this many clipped points ...
OBS_MIN_FILTERS    = 2        # ... need at least this many such filters -> "yes"
PERIOD_UNC_CONF    = 0.95     # confidence for the F-test period-uncertainty band (LS / PDM / Fourier)
PERIOD_UNC_FREQ_FRAC    = 0.01    # period-unc gate (freq-space): period uncertainty must be within this ...
PERIOD_UNC_GATE_FLOOR_H = 1e-17   # ... fraction of the period (period_unc <= FRAC*P); FLOOR (h) ~ zero -> "yes"
AMP_SNR_GATE          = 15    # amplitude-SNR gate: mbls_amplitude / amp_error >= this -> "yes"
MBLS_COLOR_MIN_OBS    = 5     # a filter needs >= this many obs for its MBLS-offset colour
# --- inline BIC / permutation-FAP (run per gate-passing period in build_agreement_rows) ---
FAP_PASS_MAX        = 0.05    # a period passes the BIC/FAP gate if FAP <= this
FAP_N_RANDOM        = 100     # permutations per object (min non-zero FAP = 1/101; raise to 1000 for final)
FAP_GRID_OVERSAMPLE = 10      # freq-grid oversampling for the FAP search (same RANGE as the LS search)
FAP_SEED            = 42      # base RNG seed; per-object offset derived from name -> reproducible
FAP_NTERMS_BASE     = 2       # match analyze_multiband's MBLS
FAP_NTERMS_BAND     = 0
# --- cross-method agreement (ported from model_agreement.ipynb) --------------------
AGREE_TOL          = 0.01                              # 1% match tolerance (in frequency, 1/P)
AGREE_PDM_SCALES   = [2.0, 0.5, 2.0 / 3.0, 3.0 / 2.0]  # one-shot PDM rescales to align w/ LS/Fourier
AGREE_DEDUP_RATIOS = [1.0, 2.0, 0.5, 3.0 / 2.0, 2.0 / 3.0]   # PDM "same period" multiples


class SkipObject(Exception):   # raised to skip an object entirely; the batch loop writes no CSV row for it
    pass


# --- JPL Small-Body Database enrichment (queried per object as its row is built) ---
SBDB_ENRICH   = True     # append orbit_class / neo / pha / rotation_period_h from JPL SBDB
SBDB_URL      = "https://ssd-api.jpl.nasa.gov/sbdb.api"
SBDB_RETRIES  = 4        # per-object retries on transient (429/5xx / network) errors
SBDB_TIMEOUT  = 30
SBDB_COLS     = ("orbit_class", "neo", "pha", "rotation_period_h")


def _sbdb_lookup(designation):
    """One JPL SBDB lookup -> {orbit_class, neo, pha, rotation_period_h}. Never raises
    (SBDB trouble must not fail the asteroid): transient errors retry with backoff, a
    genuine not-found -> orbit_class='not_found', exhausted retries -> 'error'; the four
    keys are ALWAYS present so the master-CSV schema stays consistent."""
    import requests, time
    blank = {c: "" for c in SBDB_COLS}
    if not SBDB_ENRICH:
        return dict(blank)
    for _a in range(SBDB_RETRIES):
        try:
            r = requests.get(SBDB_URL, params={"sstr": str(designation).strip(), "phys-par": "true"},
                             timeout=SBDB_TIMEOUT)
            if r.status_code == 200:
                d = r.json(); obj = d.get("object")
                if not obj:
                    return {**blank, "orbit_class": "not_found"}
                rot = ""
                for p in (d.get("phys_par") or []):
                    if p.get("name") == "rot_per":
                        rot = p.get("value", "") or ""
                        break
                return {"orbit_class": (obj.get("orbit_class") or {}).get("code", "") or "",
                        "neo": "Y" if obj.get("neo") else "N",
                        "pha": "Y" if obj.get("pha") else "N",
                        "rotation_period_h": rot}
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 * (_a + 1)); continue
            return {**blank, "orbit_class": "not_found"}
        except Exception:
            time.sleep(2 * (_a + 1))
    return {**blank, "orbit_class": "error"}
DEDUP_TOL = 0.01        # treat periods within +/- 1% (or a day-night alias) as the same
_PC_ROT = {}            # slug -> top-period multiband-LS fit, for rotation subtraction in the phase curve


def print_day_night_aliases(P_h, n_max=2):
    """Print the day-night (1 cycle/day) aliases of a period (P in hours)."""
    f0 = HOURS_PER_DAY / P_h
    print(f"  day-night aliases of P = {P_h:.4f} h  (f0 = {f0:.4f} c/d):")
    for n in range(-n_max, n_max + 1):
        if n == 0:
            continue
        f = f0 + n
        if f > 0:
            print(f"     {n:+d} c/d -> f = {f:7.4f} c/d  ->  P = {HOURS_PER_DAY / f:8.4f} h")
        else:
            print(f"     {n:+d} c/d -> f = {f:7.4f} c/d  ->  (non-physical)")


def count_model_peaks(y, prom_frac=PEAK_PROM_FRAC):
    """Number of maxima per cycle in a periodic model curve y (phase 0..1)."""
    y = np.asarray(y)
    ptp = float(y.max() - y.min())
    if ptp <= 0:
        return 0
    n = len(y)
    tiled = np.concatenate([y, y, y])               # triple-tile for wrap-around
    pk, _ = find_peaks(tiled, prominence=prom_frac * ptp)
    return int(np.sum((pk >= n) & (pk < 2 * n)))    # peaks within the middle copy


def rotation_period(P_h, n_peaks):
    """Scale a period so the folded curve has 2 maxima (asteroid convention)."""
    return P_h * 2.0 / n_peaks if n_peaks >= 1 else P_h


def _phase_coverage_ok(phase):
    """CSV-eligibility phase-coverage gate: fold the points into PHASE_COV_NBINS
    equal bins over phase [0,1); the solution is eligible only if at least
    PHASE_COV_MIN_BINS bins hold >= PHASE_COV_MIN_PER_BIN points (the same
    ">=2 points per bin" idea PDM already uses for its weights). Applied to each
    method's top 1/2/3 folds. Returns (ok, n_pass)."""
    ph = np.asarray(phase, dtype=float)
    idx = np.clip((ph * PHASE_COV_NBINS).astype(int), 0, PHASE_COV_NBINS - 1)
    counts = np.bincount(idx, minlength=PHASE_COV_NBINS)
    n_pass = int(np.count_nonzero(counts >= PHASE_COV_MIN_PER_BIN))
    return (n_pass >= PHASE_COV_MIN_BINS), n_pass


def _accept_halfwidth(objective, periods_h, j_best, crit_factor):
    """Half-width (hours) of the contiguous 'acceptable' period interval around index
    j_best: a trial period is acceptable if objective(P) <= objective[j_best]*crit_factor.
    `objective` is a residual-variance-like array (LOWER = better fit): sigma^2 (Fourier),
    coverage-penalised theta (PDM), or (1 - power) (LS). Returns (P_unc, n_regions)."""
    o = np.asarray(objective, dtype=float)
    ph = np.asarray(periods_h, dtype=float)
    thresh = o[j_best] * crit_factor
    mask = o <= thresh
    regions, i, n = [], 0, len(mask)
    while i < n:
        if mask[i]:
            k = i
            while k < n and mask[k]:
                k += 1
            regions.append((i, k - 1)); i = k
        else:
            i += 1
    main = next((r for r in regions if r[0] <= j_best <= r[1]), (j_best, j_best))
    p_lo, p_hi = sorted([ph[main[0]], ph[main[1]]])
    return 0.5 * (p_hi - p_lo), len(regions)


def _period_uncertainty(objective, periods_h, j_best, N, nparams, conf=PERIOD_UNC_CONF):
    """Approximate period uncertainty (half-width, hours) from the F-test acceptance band
    on a residual-variance-like objective -- the SAME recipe analyze_fourier uses for its
    adopted period, generalised so LS/PDM/Fourier all report a comparable +/-. `nparams` =
    model params at that period (Fourier 2k+1; LS 2*nterms_base + n_bands; PDM ~ n_bins).
    Grid-quantised and approximate. Returns (P_unc, n_regions)."""
    from scipy.stats import f as _fdist
    nu = max(1, int(N) - int(nparams))
    crit = float(_fdist.ppf(conf, nu, nu))
    return _accept_halfwidth(objective, periods_h, j_best, crit)


def _ls_top_peaks(power, n=3, periods=None, min_sep=0.01):
    """Indices of the n strongest distinct peaks (local maxima), strongest first.

    If `periods` is given, enforce a minimum FRACTIONAL separation `min_sep`
    (default 1%) between kept peaks: scanning strongest-first, a peak whose period
    is within min_sep of an already-kept (stronger) peak is SKIPPED, so near-
    duplicates never take a top slot and the next genuinely-different peak is
    promoted in their place. The stronger peak (better periodogram value) stays.
    Used identically by LS (power), PDM (-theta) and Fourier (-sigma2).
    """
    interior = np.where((power[1:-1] > power[:-2]) &
                        (power[1:-1] >= power[2:]))[0] + 1
    if interior.size == 0:
        interior = np.array([int(np.argmax(power))])
    ordered = interior[np.argsort(power[interior])[::-1]]        # strongest first
    if periods is None:
        return ordered[:n]
    kept = []
    for idx in ordered:
        p = periods[idx]
        if any(abs(p - periods[j]) / periods[j] < min_sep for j in kept):
            continue                                             # within 1% of a stronger peak -> drop
        kept.append(idx)
        if len(kept) == n:
            break
    return np.array(kept, dtype=int)


def _is_duplicate(P_raw, P_corr, seen, tol=DEDUP_TOL):
    """A top period is a duplicate of a kept one if its CORRECTED period matches
    within tol. Genuine duplicates are dropped (blanked)."""
    for r, c in seen:
        if abs(P_corr - c) / c < tol:
            return True
    return False


def _is_daynight_alias(P_raw, seen, tol=DEDUP_TOL, n_max=2):
    """True if this peak's RAW period is a day-night (n cycle/day) alias of an
    already-kept raw period (f = f_kept +/- n c/d, n = 1..n_max). These are KEPT
    (not dropped) and instead flagged in the summary 'comment' column."""
    for r, c in seen:
        f0 = HOURS_PER_DAY / r
        for n in range(1, n_max + 1):
            for sgn in (1, -1):
                f = f0 + sgn * n
                if f > 0 and abs(P_raw - HOURS_PER_DAY / f) / (HOURS_PER_DAY / f) < tol:
                    return True
    return False


FILTER_COLORS = {          # fixed colour per LSST filter -- uniform no matter how many are present
    "u": "#8856a7",        # purple
    "g": "#2ca02c",        # green
    "r": "#d62728",        # red
    "i": "#ff7f0e",        # orange
    "z": "#8c564b",        # brown
    "y": "#e377c2",        # pink
}


def _norm_band(b):
    """Filter name -> canonical letter: strip a leading 'L' (LSST 'Lg' -> 'g') and
    lowercase, so Lu,Lg,Lr,Li,Lz,Ly become u,g,r,i,z,y."""
    b = str(b).strip()
    if len(b) == 2 and b[0] in "Ll" and b[1].lower() in "ugrizy":
        b = b[1]
    return b.lower()


def session_palette(lc):
    """Sorted FILTER list + a distinct colour per filter (dynamic in the number of
    filters). Plots colour their dots / legend by band, not by observing date."""
    bands = sorted(set(np.asarray(lc["band"]).tolist()))
    cmap = [FILTER_COLORS.get(b, "#7f7f7f") for b in bands]   # fixed colour per filter (grey = unknown)
    return bands, cmap


def band_align_mag(lc):
    """Per-band offset removal for the single-series methods (Fourier, PDM) and for
    COSMETIC multiband plotting: subtract each filter's median, re-centre on the
    overall median, so g/r/i colour offsets don't inflate the combined-series
    dispersion."""
    mag = np.asarray(lc["mag"], dtype=float)
    band = np.asarray(lc["band"])
    gm = float(np.median(mag))
    out = mag.copy()
    for b in set(band.tolist()):
        m = band == b
        out[m] = mag[m] - float(np.median(mag[m])) + gm
    return out


def lsm_model_shape(lsm, t_model, freq, ubands):
    """Shared multiband LS shape for nterms_band=0 (one periodic shape + a constant
    offset per band). Evaluate every band's model curve, remove each band's vertical
    offset (its median), then average -> the common shape, free of the arbitrary
    per-band baselines that plain averaging would fold in. Returns a ~zero-median
    shape; add a baseline (e.g. median of the aligned data) when plotting."""
    models = lsm.model(t_model, freq, bands_fit=np.asarray(ubands))
    models_centered = models - np.nanmedian(models, axis=1, keepdims=True)
    return np.nanmean(models_centered, axis=0)

In [15]:
def plot_outliers(lc, removed, sessions, cmap, slug, label, outdir):
    """Two light-curve plots: outliers flagged, then removed."""
    def _plot_nights(ax):
        for c, s in zip(cmap, sessions):
            m = np.asarray(lc["band"]) == s
            ax.errorbar(lc["t"][m], lc["mag"][m], yerr=lc["err"][m],
                        fmt="o", ms=5, lw=0.6, color=c, alpha=0.8,
                        label=s)
        ax.invert_yaxis()
        ax.set_xlabel("Light-time corrected JD (days)")
        ax.set_ylabel("Absolute magnitude (H)")
        ax.legend(fontsize=16, ncol=1, title="filter", loc="best", markerscale=2)


    # (1) light curve with the outliers shown as red x's.
    LC_FLAG_PNG = os.path.join(outdir, f"{slug}_01_lightcurve_outliers.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    _plot_nights(ax)
    if len(removed["t"]):
        ax.plot(removed["t"], removed["mag"], "x", color="red", ms=8, mew=1.5,
                label=f"removed outlier ({len(removed['t'])})")
        ax.legend(fontsize=16, ncol=1, title="filter", loc="best", markerscale=2)
    ax.set_title(f"{label} -- light curve with outliers flagged")
    fig.tight_layout()
    fig.savefig(LC_FLAG_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=LC_FLAG_PNG))

    # (2) same light curve with the outliers removed.
    LC_CLEAN_PNG = os.path.join(outdir, f"{slug}_02_lightcurve_clean.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    _plot_nights(ax)
    ax.set_title(f"{label} -- light curve")
    fig.tight_layout()
    fig.savefig(LC_CLEAN_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=LC_CLEAN_PNG))

In [16]:
def analyze_multiband(lc, sessions, cmap, slug, label, outdir, summary):
    # recompute the shared 2-10 h grid (was defined in the single-band cell)
    _f_min = 1.0 / (P_MAX_HOURS / HOURS_PER_DAY)
    _f_max = 1.0 / (P_MIN_HOURS / HOURS_PER_DAY)
    _n_periods = max(2, int(round((_f_max - _f_min)
                                 * (lc["t"].max() - lc["t"].min()) * 100)))
    _freq = np.linspace(_f_min, _f_max, _n_periods)
    ls_periods_h = (1.0 / _freq) * HOURS_PER_DAY

    from astropy.timeseries import LombScargleMultiband

    bands = np.array(lc["band"])
    _ubands = sorted(set(bands.tolist()))
    print("filters present:", _ubands)

    # LS analysis stays on RAW per-band mags (LombScargleMultiband models the band
    # offsets itself). For the FOLD PLOTS only, show band-aligned points so every
    # filter overlays the single-band model -- purely cosmetic, not fed to the fit.
    _mag_plot = band_align_mag(lc)

    # Multiband Lomb-Scargle, 2 base harmonics (reuses _freq, ls_periods_h).
    lsm = LombScargleMultiband(lc["t"], lc["mag"], bands, dy=lc["err"],
                               nterms_base=2, nterms_band=0)
    lsm_power = lsm.power(_freq)
    lsm_top_idx = _ls_top_peaks(lsm_power, 3, ls_periods_h)
    lsm_top_h = [ls_periods_h[k] for k in lsm_top_idx]
    print("Multiband Lomb-Scargle (nterms_base=2) top 3 periods (h):",
          [f"{p:.5f}" for p in lsm_top_h])

    # Periodogram with the top 3 peaks marked.
    LSM_PGRAM_PNG = os.path.join(outdir, f"{slug}_06_lombscargle_multiband.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(ls_periods_h, lsm_power, lw=0.8, color="darkgreen")
    for rank, k in enumerate(lsm_top_idx):
        c = ["red", "darkorange", "green"][rank]
        ax.axvline(ls_periods_h[k], color=c, ls="--", lw=1.2,
                   label=f"#{rank+1} = {ls_periods_h[k]:.2f} h (power={lsm_power[k]:.2f})")
    ax.set_xlabel("Trial rotation period (hours)")
    ax.set_ylabel("Multiband Lomb-Scargle power")
    ax.set_title(f"{label} -- multiband Lomb-Scargle")
    ax.legend()
    fig.tight_layout()
    fig.savefig(LSM_PGRAM_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=LSM_PGRAM_PNG))

    # Fold each top period: count model peaks, correct, refit, record.
    _t0 = lc["t"].min()
    _grid = np.linspace(0, 1, 400, endpoint=False)
    _seen_lsm = []
    for rank, P_raw in enumerate(lsm_top_h, start=1):
        if rank > 1 and lsm_power[lsm_top_idx[rank - 1]] < 0.98 * lsm_power[lsm_top_idx[0]]:
            for _c in ("_raw", "_npeaks", "", "_power_raw", "_amp"):
                summary[f"ls2_P{rank}{_c}"] = ""
            print(f"  #{rank}: LS power not within 2% of P1 -- blanked")
            continue
        f_raw = HOURS_PER_DAY / P_raw
        m_raw = lsm_model_shape(lsm, _t0 + _grid / f_raw, f_raw, _ubands)  # offset-free shared shape
        n_peaks = count_model_peaks(m_raw)
        P_corr = rotation_period(P_raw, n_peaks)

        f_cpd = HOURS_PER_DAY / P_corr
        phase = np.mod((lc["t"] - _t0) * f_cpd, 1.0)
        _shape = lsm_model_shape(lsm, _t0 + _grid / f_cpd, f_cpd, _ubands)
        model_vals = _shape + np.nanmedian(_mag_plot)   # baseline on the band-aligned data
        amp = model_vals.max() - model_vals.min()

        if amp > AMP_MAX_MAG:                               # unphysical amplitude -> reject (blank)
            for _c in ("_raw", "_npeaks", "", "_power_raw", "_amp"):
                summary[f"ls2_P{rank}{_c}"] = ""
            print(f"  #{rank}: P={P_corr:.4f} h amplitude {amp:.2f} > {AMP_MAX_MAG:g} mag -- rejected")
            continue
        if _is_duplicate(P_raw, P_corr, _seen_lsm):
            for _c in ("_raw", "_npeaks", "", "_power_raw", "_amp"):
                summary[f"ls2_P{rank}{_c}"] = ""
            print(f"  #{rank}: P={P_corr:.4f} h is a duplicate of an earlier top period -- blanked & fold skipped")
            continue
        if _is_daynight_alias(P_raw, _seen_lsm):            # keep the alias, just flag it
            _n = f"ls2_P{rank} alias"
            summary["comment"] = (summary["comment"] + "; " + _n) if summary.get("comment") else _n
            print(f"  #{rank}: P={P_corr:.4f} h is a day-night alias of an earlier top period -- kept & flagged in 'comment'")
        _seen_lsm.append((P_raw, P_corr))

        print_day_night_aliases(P_corr)
        print(f"  #{rank}: raw P={P_raw:.4f} h, model peaks={n_peaks} -> corrected P={P_corr:.4f} h")
        summary[f"ls2_P{rank}_raw"] = round(float(P_raw), 5)
        summary[f"ls2_P{rank}_npeaks"] = int(n_peaks)
        summary[f"ls2_P{rank}"] = round(float(P_corr), 5)
        summary[f"ls2_P{rank}_power_raw"] = round(float(lsm_power[lsm_top_idx[rank - 1]]), 4)
        summary[f"ls2_P{rank}_amp"] = round(float(amp), 4)
        # approx period uncertainty: F-test band on (1 - power) (LS power = 1 - chi2/chi2_ref)
        _uncr, _ = _period_uncertainty(1.0 - lsm_power, ls_periods_h, lsm_top_idx[rank - 1],
                                       len(lc["t"]), 2 * 2 + len(_ubands))   # 2 base harmonics + per-band offsets
        summary[f"ls2_P{rank}_unc"] = round(float(_uncr * (P_corr / P_raw)), 5)
        if rank == 1:
            # stash the top-period multiband-LS fit so the phase curve can subtract rotation
            _PC_ROT[slug] = {"lsm": lsm, "f_cpd": f_cpd, "ubands": _ubands, "P": P_corr}

        def _plot_lsm(P_show, phase_show, model_show, tag, title, fname):
            amp_show = model_show.max() - model_show.min()
            fig, ax = plt.subplots(figsize=(9, 6))
            for c, s in zip(cmap, sessions):
                m = np.asarray(lc["band"]) == s
                ax.errorbar(phase_show[m], _mag_plot[m], yerr=lc["err"][m],
                            fmt="o", ms=5, lw=0.4, color=c, alpha=0.7, label=s)
            ax.plot(_grid, model_show, "-", color="black", lw=2,
                    label="_nolegend_")
            ax.text(0.98, 0.96, f"amplitude = {amp_show:.2f} mag", transform=ax.transAxes,
                    ha="right", va="top", fontsize=16,
                    bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.8))
            ax.invert_yaxis()
            ax.set_xlabel("Rotational phase")
            ax.set_ylabel("Absolute magnitude (H)")
            ax.set_title(title)
            ax.legend()
            fig.tight_layout()
            fig.savefig(fname, dpi=130)
            plt.close(fig)
            display(Image(filename=fname))

        # Two folds: (1) at the UNCORRECTED raw period, (2) at the CORRECTED period.
        phase_raw = np.mod((lc["t"] - _t0) * f_raw, 1.0)
        model_raw = m_raw + np.nanmedian(_mag_plot)
        _plot_lsm(P_raw, phase_raw, model_raw, "uncorrected",
                  f"{label} -- multiband LS fold: P = {P_raw:.2f} h",
                  os.path.join(outdir, f"{slug}_07_lsm_fold_top{rank}_raw.png"))
        _plot_lsm(P_corr, phase, model_vals, "corrected",
                  f"{label} -- multiband LS fold: P = {P_corr:.2f} h",
                  os.path.join(outdir, f"{slug}_07_lsm_fold_top{rank}_corr.png"))

In [18]:
def analyze_pdm(lc, sessions, cmap, slug, label, outdir, summary):
    from PyAstronomy.pyTiming import pyPDM
    from IPython.display import Image, display
    import pandas as pd

    # PDM collapses all filters into ONE magnitude series, so remove per-band colour
    # offsets first (each band's median -> overall median). LS multiband stays raw.
    lc = dict(lc)
    lc["mag"] = band_align_mag(lc)

    # === PDM binning knobs -- change these freely ==============================
    PDM_NBINS  = 20      # number of phase bins
    PDM_COVERS = 3       # number of overlapping bin sets ("covers"); 1 = no overlap
    # ===========================================================================

    # PDM over the SAME 2-10 h window, scanned uniformly in frequency (like the other
    # two methods). Stellingwerf PDM with PDM_NBINS phase bins and PDM_COVERS covers.
    _pdm_fmin = 1.0 / (P_MAX_HOURS / HOURS_PER_DAY)
    _pdm_fmax = 1.0 / (P_MIN_HOURS / HOURS_PER_DAY)
    # Use the SAME frequency grid as the multiband Lomb-Scargle: identical range AND
    # identical sample count (LS uses np.linspace(f_min, f_max, _n_periods); mirror
    # _n_periods here). The +0.5*dVal on maxVal guarantees the endpoint is included so
    # the scanner yields exactly _n_periods points on the same frequencies as LS.
    _n_periods = max(2, int(round((_pdm_fmax - _pdm_fmin)
                                  * (lc["t"].max() - lc["t"].min()) * 100/10)))
    _dVal = (_pdm_fmax - _pdm_fmin) / (_n_periods - 1)
    _scanner = pyPDM.Scanner(minVal=_pdm_fmin, maxVal=_pdm_fmax + 0.5 * _dVal,
                             dVal=_dVal, mode="frequency")
    _pdm = pyPDM.PyPDM(lc["t"], lc["mag"])
    pdm_freq, pdm_theta = _pdm.pdmEquiBinCover(PDM_NBINS, PDM_COVERS, _scanner)
    pdm_freq = np.asarray(pdm_freq)
    pdm_theta = np.asarray(pdm_theta)
    pdm_periods_h = (1.0 / pdm_freq) * HOURS_PER_DAY

    # --- phase-coverage penalty -------------------------------------------------
    # At each trial period, count how many of the PDM_NBINS phase bins hold >1 point
    # (PDM only uses bins with >=2 points). A poorly-covered fold (few filled bins ->
    # missing data) gets theta scaled UP by PDM_NBINS / filled  (>=1; e.g. 16/20 filled
    # -> x1.25), since larger theta is worse. Peaks are then picked from this MODIFIED
    # periodogram so under-sampled periods can't win; the raw theta is kept unchanged
    # for reporting. Vectorised in frequency chunks (cost ~ a few % of the pyPDM scan).
    _cov_t = np.asarray(lc["t"], dtype=float) - float(np.min(lc["t"]))
    _cov_pen = np.empty(pdm_freq.size, dtype=float)
    _CH = 4096
    for _s in range(0, pdm_freq.size, _CH):
        _fc = pdm_freq[_s:_s + _CH]
        _bi = (np.mod(_cov_t[:, None] * _fc[None, :], 1.0) * PDM_NBINS).astype(np.int32)
        np.minimum(_bi, PDM_NBINS - 1, out=_bi)
        _comb = (_bi + PDM_NBINS * np.arange(_fc.size, dtype=np.int32)[None, :]).ravel()
        _cnt = np.bincount(_comb, minlength=PDM_NBINS * _fc.size).reshape(_fc.size, PDM_NBINS)
        _cov_pen[_s:_s + _CH] = PDM_NBINS / np.maximum(np.count_nonzero(_cnt >= 2, axis=1), 1)
    pdm_theta_mod = pdm_theta * _cov_pen

    # PDM MINIMISES theta -> deepest dips are best. Peaks are found on the coverage-
    # penalised theta (peaks of -theta_mod are its minima).
    pdm_top_idx = _ls_top_peaks(-pdm_theta_mod, 3, pdm_periods_h)
    pdm_top_idx_orig = _ls_top_peaks(-pdm_theta, 3, pdm_periods_h)   # raw-theta peaks: raw periodogram plot only
    pdm_top_h = [pdm_periods_h[k] for k in pdm_top_idx]
    print("PDM top 3 periods (h):", [f"{p:.5f}" for p in pdm_top_h],
          "theta:", [f"{pdm_theta[k]:.3f}" for k in pdm_top_idx],
          "theta*:", [f"{pdm_theta_mod[k]:.3f}" for k in pdm_top_idx])

    # Periodogram 1: the ORIGINAL theta, with the chosen minima marked.
    PDM_PGRAM_PNG = os.path.join(outdir, f"{slug}_09_pdm.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(pdm_periods_h, pdm_theta, lw=0.8, color="purple")
    for rank, k in enumerate(pdm_top_idx_orig):
        c = ["red", "darkorange", "green"][rank]
        ax.axvline(pdm_periods_h[k], color=c, ls="--", lw=1.2,
                   label=f"#{rank+1} = {pdm_periods_h[k]:.2f} h (theta={pdm_theta[k]:.2f})")
    ax.set_xlabel("Trial rotation period (hours)")
    ax.set_ylabel("PDM theta")
    ax.set_title(f"{label} -- Phase Dispersion Minimization")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PDM_PGRAM_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=PDM_PGRAM_PNG))

    # Periodogram 2: the coverage-penalised theta actually used to pick the peaks.
    PDM_PGRAM_MOD_PNG = os.path.join(outdir, f"{slug}_09_pdm_penalised.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(pdm_periods_h, pdm_theta_mod, lw=0.8, color="teal")
    for rank, k in enumerate(pdm_top_idx):
        c = ["red", "darkorange", "green"][rank]
        ax.axvline(pdm_periods_h[k], color=c, ls="--", lw=1.2,
                   label=f"#{rank+1} = {pdm_periods_h[k]:.2f} h (theta*={pdm_theta_mod[k]:.2f})")
    ax.set_xlabel("Trial rotation period (hours)")
    ax.set_ylabel("PDM theta (coverage-penalised)")
    ax.set_title(f"{label} -- PDM (coverage-penalised)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PDM_PGRAM_MOD_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=PDM_PGRAM_MOD_PNG))


    def _pdm_cover_bin_means(phase, mag, nbins=PDM_NBINS, covers=PDM_COVERS):
        """
        Reconstruct the phase-shifted bin sets used by pdmEquiBinCover, for ANY
        nbins/covers.  Returns one row per (non-empty) PDM bin:
            cover, bin, start, end, center, mean, std, n
        """
        width = 1.0 / nbins
        offset_step = 1.0 / (nbins * covers)

        rows = []
        phase = np.asarray(phase)
        mag = np.asarray(mag)

        for cover in range(covers):
            offset = cover * offset_step
            # Shift phase so this cover's bins are normal [0, width), [width, 2width)...
            shifted_phase = (phase - offset) % 1.0
            bin_idx = np.clip(np.floor(shifted_phase / width).astype(int), 0, nbins - 1)

            for b in range(nbins):
                sel = bin_idx == b
                n = int(sel.sum())
                if n < 2:                       # PDM neglects bins with < 2 points
                    continue
                start = (offset + b * width) % 1.0
                end = (offset + (b + 1) * width) % 1.0
                center = (offset + (b + 0.5) * width) % 1.0
                rows.append({
                    "cover": cover, "bin": b, "start": start, "end": end,
                    "center": center, "mean": mag[sel].mean(),
                    "std": mag[sel].std(ddof=1), "n": n,
                })

        # explicit columns so an empty result still has a usable frame
        return pd.DataFrame(rows, columns=["cover", "bin", "start", "end",
                                           "center", "mean", "std", "n"])


    def _draw_phase_bin_segment(ax, start, end, y, **kwargs):
        """Draw a horizontal bin-mean segment on cyclic phase 0-1 (handles wrap)."""
        if start < end:
            ax.hlines(y, start, end, **kwargs)
        else:
            ax.hlines(y, start, 1.0, **kwargs)
            ax.hlines(y, 0.0, end, **kwargs)


    def _cover_colors(n):
        """A distinct colour per cover, for ANY number of covers."""
        return plt.cm.Greys(np.linspace(0.4, 0.95, max(int(n), 1)))


    def plot_pdm_fold(P_h, title, out_path):
        """Folded light curve at P_h with the PDM cover-bin means overlaid.

        Works for any PDM_NBINS / PDM_COVERS; if the bins are too sparse to form
        any valid (>=2 point) bin, it still renders the folded data alone.
        """
        f_cpd = HOURS_PER_DAY / P_h
        phase = np.mod((lc["t"] - lc["t"].min()) * f_cpd, 1.0)
        pdm_bins = _pdm_cover_bin_means(phase, np.asarray(lc["mag"]),
                                        nbins=PDM_NBINS, covers=PDM_COVERS)
        have = len(pdm_bins) > 0
        amp = (pdm_bins["mean"].max() - pdm_bins["mean"].min()) if have else float("nan")

        fig, ax = plt.subplots(figsize=(9, 6))
        for c, s in zip(cmap, sessions):
            m = np.asarray(lc["band"]) == s
            ax.errorbar(phase[m], lc["mag"][m], yerr=lc["err"][m],
                        fmt="o", ms=5, lw=0.4, color=c, alpha=0.7, label=s)

        colors = _cover_colors(PDM_COVERS)
        for cover in range(PDM_COVERS):
            sub = pdm_bins[pdm_bins["cover"] == cover] if have else pdm_bins
            if len(sub) == 0:
                continue
            first = True
            for _, row in sub.iterrows():
                _draw_phase_bin_segment(ax, row["start"], row["end"], row["mean"],
                                        color=colors[cover], lw=2.0, alpha=0.75,
                                        label=None)
                first = False
            ax.scatter(sub["center"], sub["mean"], color=colors[cover], s=32, alpha=0.9)

        amp_txt = f"{amp:.2f} mag" if np.isfinite(amp) else "n/a (bins too sparse)"
        ax.text(0.98, 0.96, f"PDM cover-bin amplitude = {amp_txt}",
                transform=ax.transAxes, ha="right", va="top", fontsize=17,
                bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.8))
        ax.invert_yaxis()
        ax.set_xlabel("Rotational phase")
        ax.set_ylabel("Absolute magnitude (H)")
        ax.set_title(title)
        if have:
            ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=130)
        plt.close(fig)
        display(Image(filename=out_path))


    def pdm_curve(P_h, nbins=PDM_NBINS, covers=PDM_COVERS):
        """Peak-counting / amplitude model that MATCHES the pdmEquiBinCover scan:
        the SAME cover-bin means (nbins bins x covers, <2-point bins neglected exactly
        as PDM does) that the periodogram minimises and plot_pdm_fold draws
        -- interpolated onto a uniform phase grid and averaged over the covers. (The old
        version used a single cover, kept singleton bins, and interpolated empties, so it
        was not the model the PDM minimum was found with.)"""
        f = HOURS_PER_DAY / P_h
        phase = np.mod((lc["t"] - lc["t"].min()) * f, 1.0)
        bins = _pdm_cover_bin_means(phase, np.asarray(lc["mag"]),
                                    nbins=nbins, covers=covers)
        grid = np.linspace(0, 1, 400, endpoint=False)
        cover_curves = []
        for cover in range(covers):
            sub = bins[bins["cover"] == cover]
            if len(sub) < 2:                          # need >=2 filled bins to interpolate
                continue
            xs = np.asarray(sub["center"]); ys = np.asarray(sub["mean"])
            order = np.argsort(xs); xs, ys = xs[order], ys[order]
            cover_curves.append(np.interp(grid, np.concatenate([xs - 1, xs, xs + 1]),
                                          np.concatenate([ys, ys, ys])))
        if not cover_curves:                          # bins too sparse to model
            return np.full_like(grid, np.nan)
        return np.nanmean(np.vstack(cover_curves), axis=0)

    # Fold each top period AS FOUND -- no peak adjustment (the PDM top period is used directly).
    _seen_pdm = []
    for rank, P in enumerate(pdm_top_h, start=1):
        if rank > 1 and pdm_theta_mod[pdm_top_idx[rank - 1]] > 1.02 * pdm_theta_mod[pdm_top_idx[0]]:
            for _c in ("", "_theta", "_amp", "_unc", "_coverage_pct"):
                summary[f"pdm_P{rank}{_c}"] = ""
            print(f"  #{rank}: PDM theta not within 2% of P1 -- blanked")
            continue
        _curve = pdm_curve(P)
        amp = float(np.nanmax(_curve) - np.nanmin(_curve))

        if np.isfinite(amp) and amp > AMP_MAX_MAG:          # unphysical amplitude -> reject (blank)
            for _c in ("", "_theta", "_amp", "_unc", "_coverage_pct"):
                summary[f"pdm_P{rank}{_c}"] = ""
            print(f"  #{rank}: P={P:.4f} h amplitude {amp:.2f} > {AMP_MAX_MAG:g} mag -- rejected")
            continue
        if _is_duplicate(P, P, _seen_pdm):
            for _c in ("", "_theta", "_amp", "_unc", "_coverage_pct"):
                summary[f"pdm_P{rank}{_c}"] = ""
            print(f"  #{rank}: P={P:.4f} h is a duplicate of an earlier top period -- blanked & fold skipped")
            continue
        if _is_daynight_alias(P, _seen_pdm):                # flag day-night aliases in the comment
            _n = f"pdm_P{rank} alias"
            summary["comment"] = (summary["comment"] + "; " + _n) if summary.get("comment") else _n
            print(f"  #{rank}: P={P:.4f} h is a day-night alias of an earlier top period -- kept & flagged in 'comment'")
        _seen_pdm.append((P, P))

        print_day_night_aliases(P)
        print(f"  #{rank}: PDM period P={P:.4f} h (used as found, no peak adjustment)")
        summary[f"pdm_P{rank}"] = round(float(P), 5)
        summary[f"pdm_P{rank}_theta"] = round(float(pdm_theta_mod[pdm_top_idx[rank - 1]]), 4)   # weighted (coverage-penalized) theta
        summary[f"pdm_P{rank}_amp"] = round(float(amp), 4)
        # approx period uncertainty: F-test band on the (coverage-penalised) theta periodogram
        _uncr, _ = _period_uncertainty(pdm_theta_mod, pdm_periods_h, pdm_top_idx[rank - 1],
                                       len(lc["t"]), PDM_NBINS)
        summary[f"pdm_P{rank}_unc"] = round(float(_uncr), 5)   # PDM period is used as-found (no peak-doubling) -> corrected == raw
        _cov_ok, _cov_n = _phase_coverage_ok(np.mod((lc["t"] - lc["t"].min()) * (HOURS_PER_DAY / P), 1.0))
        summary[f"pdm_P{rank}_coverage_pct"] = round(100.0 * _cov_n / PHASE_COV_NBINS, 1)   # % of 20 bins with >=2 pts

        plot_pdm_fold(P,
                      f"{label} -- PDM fold: P={P:.2f} h",
                      os.path.join(outdir, f"{slug}_10_pdm_fold_top{rank}.png"))

In [19]:
def analyze_fourier(lc, sessions, cmap, slug, label, outdir, summary):
    from scipy.stats import f as f_dist
    from scipy.signal import find_peaks
    from IPython.display import Image, display

    # === order range + thresholds ==============================================
    K_MIN, K_MAX = 2, 4         # Fourier orders (number of harmonics) to test
    FORCE_ORDER = None             # pin Fourier order to this k; set to None to restore adaptive K_MIN..K_MAX
    if FORCE_ORDER is not None:
        K_MIN = K_MAX = FORCE_ORDER
    ALPHA = 0.1                 # F-test significance for order selection
    MIN_CHI2_IMPROVE = 0     # a higher order must ALSO cut chi2 by >= this fraction
    N_CAND = 5                  # top-N dips from EACH order's periodogram
    CONF = 0.95                 # confidence level for the period-uncertainty band
    REL_NARROW, REL_BROAD = 0.02, 0.10   # accepted-interval width (fraction of P)
    VE_MIN = 0.05               # min variance-explained for a real detection
    # ============================================================================

    # Fourier fits ONE series across all filters -> band-align first (LS stays raw).
    _t_all, _m_all, _e_all = lc["t"], band_align_mag(lc), lc["err"]

    # Fourier period grid: same dynamic step count PDM uses (100/4 density),
    # from the full clipped light curve -- replaces the old fixed 20000-step grid.
    _f_min = 1.0 / (P_MAX_HOURS / HOURS_PER_DAY)
    _f_max = 1.0 / (P_MIN_HOURS / HOURS_PER_DAY)
    N_STEPS_F = max(2, int(round((_f_max - _f_min)
                                 * (lc["t"].max() - lc["t"].min()) * 100/3)))

    def _fit_chi2(tt, mm, ee, P_h, k):
        """Weighted chi2 and n_par of an order-k fit at a FIXED period (lstsq)."""
        ph = fold_phase(tt, P_h / HOURS_PER_DAY, tt.min())
        A = fourier_design(ph, k)
        w = 1.0 / ee
        cf, *_ = np.linalg.lstsq(A * w[:, None], mm * w, rcond=None)
        return float(np.sum(((mm - A @ cf) / ee) ** 2)), 2 * k + 1

    def _select_order(tt, mm, ee, P_h, N):
        """Forward selection: add the next harmonic only if it is BOTH F-test
        significant (p<ALPHA) AND cuts chi2 by >= MIN_CHI2_IMPROVE (per step). Stop
        at the first harmonic that fails either test. Returns (k, chi2 at k)."""
        chi2, npar = {}, {}
        for k in range(K_MIN, K_MAX + 1):
            chi2[k], npar[k] = _fit_chi2(tt, mm, ee, P_h, k)
        k = K_MIN
        while k < K_MAX:
            imp = chi2[k] - chi2[k + 1]
            if imp <= 0:
                break
            frac = imp / chi2[k]
            d1, d2 = npar[k + 1] - npar[k], N - npar[k + 1]
            p = f_dist.sf((imp / d1) / (chi2[k + 1] / d2), d1, d2)
            if p < ALPHA and frac >= MIN_CHI2_IMPROVE:     # next harmonic earns its place
                k += 1
            else:
                break
        return k, chi2[k]

    def _analyze_fourier(tt, mm, ee):
        """Per-order sigma search -> candidate periods pooled from EVERY order's
        periodogram (k=K_MIN..K_MAX, de-duplicated) -> parsimonious k per candidate ->
        pick the candidate with the lowest sigma AT ITS OWN order (so neither a wrong
        period nor an over-fit high k can win)."""
        N = len(tt); t0 = tt.min(); Sp = float(np.sum(1.0 / ee ** 2))
        order = {}
        for k in range(K_MIN, K_MAX + 1):
            ph_, s2_, Pk_, s2k_ = run_fourier_search(tt, mm, ee, P_MIN_HOURS, P_MAX_HOURS, N_STEPS_F, k)
            order[k] = {"periods_h": ph_, "sigma2": s2_, "Pk": Pk_,
                        "sigma2_min": s2k_, "nparams": 2 * k + 1}
        # Candidates: top-N_CAND dips from EACH order (2..K_MAX) -> up to 15. Each keeps the ORDER
        # it was found at and its power there (sigma at that order). Shared grid -> a dip index is
        # the same period at every order.
        pool = []
        for kk in range(K_MIN, K_MAX + 1):
            for i in _ls_top_peaks(-order[kk]["sigma2"], N_CAND, order[kk]["periods_h"]):
                pool.append({"i": int(i), "k": kk, "P": float(order[kk]["periods_h"][i]),
                             "sigma": float(np.sqrt(order[kk]["sigma2"][i]))})

        # Step 1 -- de-duplicate within 1% (raw period), ALWAYS keeping the LOWEST order.
        pool.sort(key=lambda d: (d["k"], d["sigma"]))
        ded = []
        for c in pool:
            if not any(abs(c["P"] - q["P"]) / q["P"] < 0.01 for q in ded):
                ded.append(c)

        # F-test (paper form) between TWO candidates at DIFFERENT periods/orders: does the higher-
        # order one (hi) significantly beat the lower-order one (lo)?
        #   p = CDF_F(sigma2_hi / sigma2_lo ; nu_lo, nu_hi) ;  hi wins if p < ALPHA (10% level).
        def _beats(hi, lo):
            s2_hi = float(order[hi["k"]]["sigma2"][hi["i"]])
            s2_lo = float(order[lo["k"]]["sigma2"][lo["i"]])
            if s2_lo <= 0 or s2_hi >= s2_lo:
                return False
            nu_hi, nu_lo = N - (2 * hi["k"] + 1), N - (2 * lo["k"] + 1)
            return f_dist.cdf(s2_hi / s2_lo, nu_lo, nu_hi) < ALPHA

        # Step 2 -- pick 3, RESTARTING FROM THE TOP each time. Scan candidates highest-power first;
        # the first VALID one is USED (selected) and REMOVED, then we restart the scan from the top.
        # Valid = an order-K_MIN (order-2) candidate (nothing below it), OR a higher-order candidate that
        # BEATS the highest-power lower-order candidate STILL in the list by the F-test. A candidate that
        # isn't valid this round is KEPT (skipped) -- so once a stronger period is removed it is re-checked
        # against a weaker baseline. A pick whose 2-peak-aligned period duplicates an already-chosen one
        # (<1%) is dropped, then we re-scan.
        _cgrid = np.linspace(0, 1, 400, endpoint=False)
        remaining = sorted(ded, key=lambda d: d["sigma"])      # highest power first
        cands, dup, rej = [], [], []
        while len(cands) < 3:
            chosen = None
            for cur in remaining:                              # scan from the top
                if cur["k"] == K_MIN:
                    chosen = cur; break
                lowers = [q for q in remaining if q["k"] < cur["k"]]
                lb = min(lowers, key=lambda q: q["sigma"]) if lowers else None
                if lb is None or _beats(cur, lb):
                    chosen = cur; break                        # first valid candidate
                # else: not valid this round -> skip, keep for a later round
            if chosen is None:
                break                                          # nothing valid remains
            remaining.remove(chosen)
            _, _m0 = fit_fourier(fold_phase(tt, chosen["P"] / HOURS_PER_DAY, t0), mm, ee, chosen["k"])
            _Pc = rotation_period(chosen["P"], count_model_peaks(_m0(_cgrid)))
            _, _mc = fit_fourier(fold_phase(tt, _Pc / HOURS_PER_DAY, t0), mm, ee, chosen["k"])
            if float(np.ptp(_mc(_cgrid))) > AMP_MAX_MAG:       # unphysical folded amplitude -> reject
                rej.append(chosen)
                continue
            if any(abs(_Pc - q["_Pc"]) / q["_Pc"] < 0.01 for q in cands):
                dup.append(chosen)                             # a multiple/duplicate of a chosen period
                continue                                       # drop, try next
            chosen["_Pc"] = _Pc
            cands.append(chosen)
        if not cands:                                          # degenerate fallback
            _im = int(np.argmin(order[K_MIN]["sigma2"]))
            cands = [{"i": _im, "k": K_MIN, "P": float(order[K_MIN]["periods_h"][_im]),
                      "sigma": float(np.sqrt(order[K_MIN]["sigma2"][_im])), "_Pc": 1.0}]
        cands = [{"P": c["P"], "k": c["k"], "sigma": c["sigma"]} for c in cands]
        win = cands[0]                                         # cands already sorted best-first
        P_best, k_best = win["P"], win["k"]
        s2 = order[k_best]["sigma2"]; periods_h = order[k_best]["periods_h"]
        jbest = int(np.argmin(np.abs(periods_h - P_best)))
        _, model = fit_fourier(fold_phase(tt, P_best / HOURS_PER_DAY, t0), mm, ee, k_best)
        return {"order": order, "N": N, "Sp": Sp, "t0": t0, "cands": cands,
                "all_cands": [{"P": c["P"], "k": c["k"], "sigma": c["sigma"]} for c in ded],
                "dup_cands": [{"P": c["P"], "k": c["k"], "sigma": c["sigma"]} for c in dup],
                "rej_cands": [{"P": c["P"], "k": c["k"], "sigma": c["sigma"]} for c in rej],
                "k_best": k_best, "s2": s2, "periods_h": periods_h, "jbest": jbest,
                "P_best": P_best, "s2_min": float(s2[jbest]), "model": model}

    # No Fourier-specific clip: fit the 5-sigma-clipped light curve as-is (like LS/PDM).
    keep = np.ones(len(_t_all), dtype=bool)
    res = _analyze_fourier(_t_all, _m_all, _e_all)

    t, mag, err = _t_all[keep], _m_all[keep], _e_all[keep]
    _sess = lc["session"][keep]
    N, Sp, t0 = res["N"], res["Sp"], res["t0"]
    order = res["order"]; k_best = res["k_best"]
    print(f"{N} points used for the Fourier solution (5-sigma-clipped light curve).")

    print("\nPer-order min-sigma period (independent scan):")
    for k in range(K_MIN, K_MAX + 1):
        print(f"  k={k}: Pk={order[k]['Pk']:.5f} h   sigma={np.sqrt(order[k]['sigma2_min']):.5f}")
    print(f"Candidates (top-{N_CAND}/order, deduped <1% to lowest order; higher order kept only if it beats the highest-power lower-order candidate by F-test p<{ALPHA}; corrected periods deduped <1%):")
    for cinfo in res["cands"]:
        print(f"    P={cinfo['P']:8.4f} h  ->  k={cinfo['k']}   sigma={cinfo['sigma']:.5f}")
    print(f"Winner (lowest sigma at its own order): P = {res['P_best']:.5f} h  at k = {k_best}")

    # flat (weighted-mean) baseline dispersion, for the variance-explained check.
    _wmean = np.sum(mag / err ** 2) / Sp
    _chi2_flat = float(np.sum(((mag - _wmean) / err) ** 2))

    # ADOPTED PERIOD + (APPROXIMATE) UNCERTAINTY at the chosen order, relative to
    # the winning period: acceptable P have sigma^2(P) <= sigma^2(P_best)*F_crit.
    s2 = res["s2"]; periods_h = res["periods_h"]
    nu = N - order[k_best]["nparams"]
    F_crit = float(f_dist.ppf(CONF, nu, nu))
    s2_min = res["s2_min"]
    s2_thresh = s2_min * F_crit
    jbest = res["jbest"]
    P_best = res["P_best"]

    mask = s2 <= s2_thresh
    regions = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            regions.append((i, j - 1))
            i = j
        else:
            i += 1
    main = next(r for r in regions if r[0] <= jbest <= r[1])
    p_lo, p_hi = sorted([periods_h[main[0]], periods_h[main[1]]])
    P_unc = 0.5 * (p_hi - p_lo)
    n_regions = len(regions)

    chi2_best = s2_min * (N - order[k_best]["nparams"]) * Sp / N
    var_expl = 1.0 - chi2_best / _chi2_flat

    if var_expl < VE_MIN:
        reliability = "failed"
    elif n_regions >= 2:
        reliability = "ambiguous"
    elif (P_unc / P_best) > REL_BROAD:
        reliability = "poor"
    elif (P_unc / P_best) <= REL_NARROW:
        reliability = "reliable"
    else:
        reliability = "poor"

    print(f"\nChosen order k={k_best}: P = {P_best:.5f} +/- {P_unc:.5f} h  "
          f"(sigma={np.sqrt(s2_min):.5f})")
    print(f"  {int(CONF*100)}% acceptable interval [{p_lo:.5f}, {p_hi:.5f}] h; "
          f"{n_regions} region(s); variance explained {var_expl:.3f}")
    print(f"  reliability: {reliability.upper()}")

    # --- plot 1: sigma periodogram per order + the 95% acceptance threshold -----
    FO_PGRAM_PNG = os.path.join(outdir, f"{slug}_11_fourier_periodogram.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = plt.cm.viridis(np.linspace(0.0, 0.9, K_MAX - K_MIN + 1))
    for c, k in zip(colors, range(K_MIN, K_MAX + 1)):
        ax.plot(order[k]["periods_h"], np.sqrt(order[k]["sigma2"]), lw=0.8, color=c,
                alpha=0.85, label=f"k={k}  (P={order[k]['Pk']:.2f} h)")
    ax.set_xlabel("Trial rotation period (hours)")
    ax.set_ylabel("sigma")
    ax.set_title(f"{label} -- Fourier Periodogram")
    ax.legend(fontsize=16)
    fig.tight_layout()
    fig.savefig(FO_PGRAM_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=FO_PGRAM_PNG))

    # individual per-k sigma periodograms (one plot each, alongside the combined one above)
    for _k in range(K_MIN, K_MAX + 1):
        FO_PGRAM_K_PNG = os.path.join(outdir, f"{slug}_11_fourier_periodogram_k{_k}.png")
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.plot(order[_k]["periods_h"], np.sqrt(order[_k]["sigma2"]), lw=0.9, color="navy",
                label=f"k={_k}  (P={order[_k]['Pk']:.2f} h)")
        ax.set_xlabel("Trial rotation period (hours)")
        ax.set_ylabel("sigma")
        ax.set_title(f"{label} -- Fourier Periodogram (k={_k})")
        ax.legend(fontsize=16)
        fig.tight_layout()
        fig.savefig(FO_PGRAM_K_PNG, dpi=130)
        plt.close(fig)
        display(Image(filename=FO_PGRAM_K_PNG))

    # --- plot 2: ALL candidate periods vs sigma (full deduped pool), top 3 ringed
    FO_ORDER_PNG = os.path.join(outdir, f"{slug}_12_fourier_order.png")
    fig, ax = plt.subplots(figsize=(7, 5.5))
    _all = res["all_cands"]
    _selP = {round(d["P"], 6) for d in res["cands"]}                 # the 3 chosen periods
    ax.scatter([d["P"] for d in _all], [d["sigma"] for d in _all],
               c="navy", s=45, zorder=3, label=f"candidates ({len(_all)})")
    _sel = [d for d in _all if round(d["P"], 6) in _selP]
    ax.scatter([d["P"] for d in _sel], [d["sigma"] for d in _sel],
               facecolors="none", edgecolors="black", s=180, lw=2, zorder=5, label="chosen top 3")
    _dupP = {round(d["P"], 6) for d in res.get("dup_cands", [])}    # dropped as a multiple/dup of a chosen period
    _dup = [d for d in _all if round(d["P"], 6) in _dupP]
    ax.scatter([d["P"] for d in _dup], [d["sigma"] for d in _dup],
               facecolors="none", edgecolors="gold", s=180, lw=2, zorder=4, label="duplicate / multiple")
    _rejP = {round(d["P"], 6) for d in res.get("rej_cands", [])}    # rejected: folded amplitude > cap
    _rej = [d for d in _all if round(d["P"], 6) in _rejP]
    ax.scatter([d["P"] for d in _rej], [d["sigma"] for d in _rej],
               facecolors="none", edgecolors="red", s=180, lw=2, zorder=4, label="rejected (amp>cap)")
    for d in _all:
        ax.annotate(f"k={d['k']}", (d["P"], d["sigma"]),
                    textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
    ax.set_xlabel("Candidate period (hours)")
    ax.set_ylabel("sigma")
    ax.set_title(f"{label} -- Fourier candidate comparison")
    fig.tight_layout()
    fig.savefig(FO_ORDER_PNG, dpi=130)
    plt.close(fig)
    display(Image(filename=FO_ORDER_PNG))

    # 5) 2-peak rotation correction (the Fourier order k is kept unchanged).
    _grid = np.linspace(0, 1, 400, endpoint=False)
    n_peaks = count_model_peaks(res["model"](_grid))
    P_corr = rotation_period(P_best, n_peaks)
    _corr_factor = P_corr / P_best                     # scale the uncertainty too
    P_unc_corr = P_unc * _corr_factor
    k_corr = k_best                                    # keep the same order through the correction
    phase_corr = fold_phase(t, P_corr / HOURS_PER_DAY, t0)
    _, model_corr = fit_fourier(phase_corr, mag, err, k_corr)
    _resid_corr = mag - model_corr(phase_corr)
    chi2_corr = float(np.sum((_resid_corr / err) ** 2))
    npar_corr = 2 * k_corr + 1
    grid = np.linspace(0, 1, 400)
    fit_vals = model_corr(grid)
    amp = fit_vals.max() - fit_vals.min()

    print_day_night_aliases(P_corr)
    print(f"Peak correction: winner P={P_best:.5f} +/- {P_unc:.5f} h (k={k_best}), "
          f"model peaks={n_peaks} -> corrected P={P_corr:.5f} +/- {P_unc_corr:.5f} h, "
          f"order kept at k={k_corr}")

    # --- top-3 candidates: peak-correct, fold-plot, and record EACH -------------
    # rank 1 reuses the adopted winner solution; ranks 2-3 are corrected & refit here.
    _cands3 = res["cands"][:3]
    _sig1 = _cands3[0]["sigma"] if _cands3 else float("nan")   # winner (P1) own-order sigma
    for _i in range(1, 4):
        _c = _cands3[_i - 1] if _i - 1 < len(_cands3) else None
        if _c is not None and _i > 1 and _c["sigma"] > 1.02 * _sig1:   # own adopted-order sigma
            print(f"  cand{_i}: sigma {_c['sigma']:.4f} not within 2% of P1 ({_sig1:.4f}) -- blanked")
            _c = None                                        # not within 2% of P1's sigma -> blank
        if _c is None:                                       # <3 cands / out-of-range -> blank, aligned cols
            for _s, _blank in (("_P", float("nan")), ("_P_raw", float("nan")), ("_k", -1),
                               ("_sigma", float("nan")), ("_npeaks", -1),
                               ("_amp", float("nan")), ("_redchi2", float("nan")),
                               ("_P_unc", float("nan"))):
                summary[f"fourier_cand{_i}{_s}"] = _blank
            continue
        if _i == 1:                                          # winner: reuse already-computed solution
            _Pc, _kc, _npk, _Pcorr = P_best, k_best, n_peaks, P_corr
            _ph, _fit, _amp, _chi2 = phase_corr, fit_vals, amp, chi2_corr
        else:                                                # correct + refit this candidate
            _Pc, _kc = _c["P"], _c["k"]
            _, _m0 = fit_fourier(fold_phase(t, _Pc / HOURS_PER_DAY, t0), mag, err, _kc)
            _npk = count_model_peaks(_m0(_grid))
            _Pcorr = rotation_period(_Pc, _npk)
            _ph = fold_phase(t, _Pcorr / HOURS_PER_DAY, t0)
            _, _mcorr = fit_fourier(_ph, mag, err, _kc)
            _fit = _mcorr(grid)
            _amp = float(_fit.max() - _fit.min())
            _chi2 = float(np.sum(((mag - _mcorr(_ph)) / err) ** 2))
        _redchi2 = _chi2 / (len(t) - (2 * _kc + 1))

        summary[f"fourier_cand{_i}_P"]       = round(float(_Pcorr), 5)   # peak-corrected period
        summary[f"fourier_cand{_i}_P_raw"]   = round(float(_Pc), 5)
        summary[f"fourier_cand{_i}_k"]       = int(_kc)
        summary[f"fourier_cand{_i}_sigma"]   = round(float(_c["sigma"]), 6)
        summary[f"fourier_cand{_i}_npeaks"]  = int(_npk)
        summary[f"fourier_cand{_i}_amp"]     = round(float(_amp), 4)
        summary[f"fourier_cand{_i}_redchi2"] = round(float(_redchi2), 4)
        # approx period uncertainty: F-test band on this candidate's own-order sigma^2 periodogram
        _uncr, _ = _period_uncertainty(order[_kc]["sigma2"], order[_kc]["periods_h"],
                                       int(np.argmin(np.abs(np.asarray(order[_kc]["periods_h"]) - _Pc))),
                                       N, 2 * _kc + 1)
        summary[f"fourier_cand{_i}_P_unc"] = round(float(_uncr * (_Pcorr / _Pc)), 5)

        # --- folded light curve at this candidate's corrected period ---
        FO_FOLD_PNG = os.path.join(outdir, f"{slug}_13_fourier_folded_top{_i}.png")
        fig, ax = plt.subplots(figsize=(9, 6))
        for c, s_ in zip(cmap, sessions):
            m = np.asarray(lc["band"])[keep] == s_
            ax.errorbar(_ph[m], mag[m], yerr=err[m], fmt="o", ms=5, lw=0.4,
                        color=c, alpha=0.7, label=s_)
        ax.plot(grid, _fit, "-", color="black", lw=2, label="_nolegend_")
        ax.text(0.98, 0.96, f"amplitude = {_amp:.2f} mag", transform=ax.transAxes,
                ha="right", va="top", fontsize=17,
                bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.8))
        ax.invert_yaxis()
        ax.set_xlabel("Rotational phase")
        ax.set_ylabel("Absolute magnitude (H)")
        ax.set_title(f"{label} -- Fourier fold: P = {_Pcorr:.2f} h (k={_kc})")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FO_FOLD_PNG, dpi=130)
        plt.close(fig)
        display(Image(filename=FO_FOLD_PNG))


In [ ]:
# ==========================================================================
#  LS RESIDUAL FOLD  (LS top-period fold with the LS model subtracted)  -- additive
# ==========================================================================
def analyze_ls_residual(lc, sessions, cmap, slug, label, outdir):
    """Folded light curve at the TOP multiband-LS period with the LS model SUBTRACTED,
    so the leftover scatter (what the phase curve keeps) is visible. Uses the same
    top-period fit the phase curve removes (_PC_ROT). Additive; skipped if the top LS
    period was unavailable/rejected."""
    from IPython.display import Image, display
    _rm = _PC_ROT.get(slug)
    if _rm is None:
        print("LS residual fold: no LS top-period model -- skipped.")
        return
    try:
        lsm = _rm["lsm"]; f_cpd = _rm["f_cpd"]; ubands = _rm["ubands"]; P = _rm["P"]
        t = np.asarray(lc["t"]); t0 = t.min()
        mag_plot = band_align_mag(lc)                        # same band-aligned points as the LS fold
        model_at_obs = lsm_model_shape(lsm, t, f_cpd, ubands) + np.nanmedian(mag_plot)
        resid = mag_plot - model_at_obs                      # data minus the LS curve
        phase = np.mod((t - t0) * f_cpd, 1.0)
        rms = float(np.sqrt(np.nanmean(resid ** 2)))

        fig, ax = plt.subplots(figsize=(9, 6))
        for c, s in zip(cmap, sessions):
            m = np.asarray(lc["band"]) == s
            ax.errorbar(phase[m], resid[m], yerr=lc["err"][m],
                        fmt="o", ms=5, lw=0.4, color=c, alpha=0.7, label=s)
        ax.axhline(0.0, color="black", lw=2, label="_nolegend_")   # the subtracted LS curve
        ax.text(0.98, 0.96, f"residual RMS = {rms:.3f} mag", transform=ax.transAxes,
                ha="right", va="top", fontsize=16,
                bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.8))
        ax.invert_yaxis()
        ax.set_xlabel("Rotational phase")
        ax.set_ylabel("Absolute magnitude (H) - LS model")
        ax.set_title(f"{label} -- LS fold residual (LS curve subtracted): P = {P:.2f} h")
        ax.legend()
        fig.tight_layout()
        out = os.path.join(outdir, f"{slug}_16_ls_residual_fold.png")
        fig.savefig(out, dpi=130)
        plt.close(fig)
        display(Image(filename=out))
    except Exception as _e:
        print(f"LS residual fold skipped: {type(_e).__name__}: {_e}")


# ==========================================================================
#  PHASE CURVE  (reduced mag vs solar phase angle; sbpy HG fit)  -- additive
# ==========================================================================
_MPC_LON = {}   # obscode -> longitude east (deg); fetched once per session


def mpc_longitude(code):
    """Longitude (deg East) for an MPC observatory code, via astroquery MPC (fetched
    once, then cached). Returns None for space/roving codes or if the lookup fails."""
    code = str(code).strip()
    if not _MPC_LON:
        import warnings as _w
        from astroquery.mpc import MPC
        with _w.catch_warnings():
            _w.simplefilter("ignore")                     # space/roving codes have masked longitudes
            for row in MPC.get_observatory_codes():
                _c = str(row["Code"]).strip()
                try:
                    _v = float(row["Longitude"])
                    if not np.isfinite(_v):
                        _v = None
                except (TypeError, ValueError):
                    _v = None
                _MPC_LON[_c] = _v
    return _MPC_LON.get(code)


def assign_nights(jd, mpc_code=None, lon_east_deg=None):
    """Group timestamps into observing NIGHTS.

    PREFERRED: use the site's LONGITUDE (looked up from its MPC observatory code) to
    convert to local solar time and split nights at local noon. This is robust for
    sparse data -- it works even with a single point per night, needs no gap, and a
    night running past local midnight stays together for any telescope/timezone.

    FALLBACK (only if no longitude is available): the data-driven daytime-gap split.

    Returns (night_index 0..K-1 in time order, split_fraction, lon_or_None, method).
    """
    jd = np.asarray(jd, dtype=float)
    if jd.size == 0:
        return np.array([], dtype=int), 0.0, None, "empty"
    lon = lon_east_deg
    if lon is None and mpc_code is not None:
        try:
            lon = mpc_longitude(mpc_code)
        except Exception as _e:
            print(f"  MPC obscode lookup failed ({type(_e).__name__}); using daytime-gap fallback.")
            lon = None
    if lon is not None:
        ljd = jd + (float(lon) / 360.0)                 # local solar JD (integer = local noon)
        night = np.floor(ljd).astype(int)               # local noon-to-noon buckets
        split = float(np.mod(-float(lon) / 360.0, 1.0)) # UT-fraction of local noon (for reporting)
        method = f"MPC {mpc_code} (lon {float(lon):.2f}E)"
    else:                                               # no longitude -> daytime-gap split
        frac = np.mod(jd, 1.0); s = np.sort(frac)
        if s.size == 1:
            split = float(np.mod(s[0] + 0.5, 1.0))
        else:
            gaps = np.append(np.diff(s), (s[0] + 1.0) - s[-1]); k = int(np.argmax(gaps))
            lo = s[k]; hi = s[k + 1] if k < s.size - 1 else s[0] + 1.0
            split = float(np.mod(0.5 * (lo + hi), 1.0))
        night = np.floor(jd - split).astype(int)
        method = "data-driven daytime-gap"
    uniq = {v: i for i, v in enumerate(sorted(set(night.tolist())))}
    return np.array([uniq[v] for v in night], dtype=int), split, lon, method


def analyze_phasecurve(lc, sessions, cmap, slug, label, outdir, summary):
    """Solar phase curve for one object.

    Y = distance-corrected reduced magnitude m_dist (BEFORE the phase correction),
        with per-band offsets removed so g/r/i overlay.
    X = solar phase angle alpha (deg).
    One nightly MEDIAN point per calendar date (median mag, median alpha).
    Fit the sbpy HG model (COBYLA, 0 <= G <= 1) -> records phase_H, phase_G.

    Fully self-contained: any failure (incl. sbpy missing) is caught, the 4 columns
    are always written, and the object's row is never lost. Additive -- nothing else
    in the pipeline is touched.
    """
    # register columns up front so a failure can never misalign the master CSV
    summary["phase_H"] = float("nan")
    summary["phase_G"] = float("nan")
    summary["phase_n_nights"] = 0
    summary["phase_rms"] = float("nan")
    summary["phase_rot_P"] = float("nan")
    summary["phase_alpha_min"] = float("nan")
    summary["phase_alpha_max"] = float("nan")
    try:
        import scipy.optimize as sco
        from sbpy.photometry import HG
        from IPython.display import Image, display

        # --- m_dist, alpha, band, date straight from the reduced CSV (not in lc) ---
        reduced_csv = os.path.join(outdir, f"{slug}_reduced.csv")
        md, al, bd, jd, jdl = [], [], [], [], []
        with open(reduced_csv, encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                try:
                    _m = float(row["m_dist"]); _a = float(row["alpha_deg"]); _j = float(row["jd"])
                    _jl = float(row["jd_ltc"])
                except (ValueError, KeyError):
                    continue
                md.append(_m); al.append(_a); jd.append(_j); jdl.append(_jl)
                bd.append(_norm_band(row.get("filter", "")))
        md = np.array(md); al = np.array(al); bd = np.array(bd); jd = np.array(jd); jdl = np.array(jdl)
        if md.size == 0:
            print("phase curve: no m_dist/alpha rows -- skipped.")
            return

        # --- rotation-subtracted reduced mag (per observation) ---
        # The LS fit was done on H_reduced (absolute mag); the zero-median rotation
        # shape it gives is subtracted here from m_dist (REDUCED mag), not from H.
        _rot_P = None
        mr = md.astype(float).copy()
        _rm = _PC_ROT.pop(slug, None)
        if _rm is not None:
            try:
                _rot = np.asarray(lsm_model_shape(_rm["lsm"], jdl, _rm["f_cpd"], _rm["ubands"]))
                mr = mr - _rot                        # zero-median rotation removed; per-filter offsets kept
                _rot_P = float(_rm["P"])
                summary["phase_rot_P"] = round(_rot_P, 5)
                print(f"phase curve: subtracted LS rotation model at top period P={_rot_P:.4f} h")
            except Exception as _e:
                print(f"phase curve: rotation subtraction skipped ({type(_e).__name__}: {_e})")
        else:
            print("phase curve: no LS top-period model -- rotation NOT subtracted")

        # --- per-filter alignment CONSTANT (same recipe as band_align_mag / Fourier / PDM):
        #     subtract each filter's median, add the overall median. Applied only in the
        #     MAIN (aligned) plot + HG fit; the by-filter plot shows the raw offsets. ---
        _gm = float(np.median(mr))
        _const = {b: _gm - float(np.median(mr[bd == b])) for b in set(bd.tolist())}
        mro = mr + np.array([_const[b] for b in bd])   # filter-aligned reduced mag

        # --- one (alpha, mag) median point per (NIGHT, FILTER) ---
        night_id, split, _lon, _method = assign_nights(jd, mpc_code=MPC_CODE)
        _bins = {}
        for nid, b, a, mu, ma in zip(night_id, bd, al, mr, mro):
            _bins.setdefault((int(nid), b), []).append((a, mu, ma))
        _keys = sorted(_bins.keys(),
                       key=lambda k: (k[1], float(np.median([r[0] for r in _bins[k]]))))
        nf_band  = np.array([k[1] for k in _keys])
        nf_alpha = np.array([float(np.median([r[0] for r in _bins[k]])) for k in _keys])
        nf_unal  = np.array([float(np.median([r[1] for r in _bins[k]])) for k in _keys])  # unaligned
        nf_algn  = np.array([float(np.median([r[2] for r in _bins[k]])) for k in _keys])  # aligned
        n_pts = len(_keys)
        n_nights = len(set(int(k[0]) for k in _keys))
        summary["phase_n_nights"] = int(n_nights)
        summary["phase_alpha_min"] = round(float(al.min()), 4)
        summary["phase_alpha_max"] = round(float(al.max()), 4)
        _uth = ((split + 0.5) % 1.0) * 24.0            # UT hour of local noon (night boundary)
        print(f"phase curve: {n_pts} (night,filter)-median point(s) over {n_nights} night(s) "
              f"[{_method}; local noon ~ {int(_uth):02d}:{int((_uth % 1) * 60):02d} UT]; "
              f"alpha {al.min():.2f}..{al.max():.2f} deg")

        # --- sbpy HG fit (COBYLA, 0 <= G <= 1) on the ALIGNED (night,filter) medians ---
        def pf(xdeg, par1, par2):
            return float(HG.evaluate(xdeg * np.pi / 180.0, par1, par2))
        def sse_fun(x, data):
            return sum((pf(d[0], x[0], x[1]) - d[1]) ** 2 for d in data)
        def fit_fun(data, x0=(6.0, 0.12)):
            cv = ({"type": "ineq", "fun": lambda x: x[1]},
                  {"type": "ineq", "fun": lambda x: 1 - x[1]})
            return sco.minimize(sse_fun, x0, args=(data,), constraints=cv, method="COBYLA")

        H_fit = G_fit = rms = float("nan"); model_ok = False
        if n_pts >= 3 and (nf_alpha.max() - nf_alpha.min()) > 1e-6:
            data = list(zip(nf_alpha.tolist(), nf_algn.tolist()))
            res = fit_fun(data, x0=(float(np.median(nf_algn)), 0.15))
            H_fit, G_fit = float(res.x[0]), float(res.x[1])
            resid = np.array([pf(a, H_fit, G_fit) for a in nf_alpha]) - nf_algn
            rms = float(np.sqrt(np.mean(resid ** 2)))
            model_ok = np.isfinite(H_fit) and np.isfinite(G_fit)
            summary["phase_H"] = round(H_fit, 4)
            summary["phase_G"] = round(G_fit, 4)
            summary["phase_rms"] = round(rms, 4)
            print(f"phase curve HG fit: H={H_fit:.3f} mag, G={G_fit:.3f}, rms={rms:.3f}")
        else:
            print("phase curve: <3 points or no phase-angle spread -- fit skipped.")

        _ylab = "Reduced magnitude" + (" (rotation-subtracted)" if _rot_P is not None else "")
        _ubset = sorted(set(nf_band.tolist()))
        _fcol = {b: FILTER_COLORS.get(b, "#7f7f7f") for b in _ubset}

        # --- plot 1 (additional): raw per-filter medians, colour = filter, NO alignment ---
        BYFILT_PNG = os.path.join(outdir, f"{slug}_18_phasecurve_byfilter.png")
        fig, ax = plt.subplots(figsize=(9, 6))
        for b in _ubset:
            m = nf_band == b
            ax.scatter(nf_alpha[m], nf_unal[m], s=55, color=_fcol[b], zorder=3, label=b)
        ax.invert_yaxis()
        ax.set_xlabel("Solar phase angle  alpha  (deg)")
        ax.set_ylabel(_ylab)
        ax.set_title(f"{label} -- phase curve by filter ({n_pts} night-filter medians, unaligned)")
        ax.legend(title="filter")
        fig.tight_layout()
        fig.savefig(BYFILT_PNG, dpi=130)
        plt.close(fig)
        display(Image(filename=BYFILT_PNG))

        # --- plot 2 (main): filter-aligned medians + HG fit, colour = filter ---
        PHASE_PNG = os.path.join(outdir, f"{slug}_17_phasecurve.png")
        fig, ax = plt.subplots(figsize=(9, 6))
        for b in _ubset:
            m = nf_band == b
            ax.scatter(nf_alpha[m], nf_algn[m], s=55, color=_fcol[b], zorder=3, label=b)
        if model_ok:
            xg = np.linspace(max(0.0, nf_alpha.min() - 1.0), nf_alpha.max() + 1.0, 200)
            ax.plot(xg, [pf(x, H_fit, G_fit) for x in xg], "-", color="red", lw=2,
                    label=f"HG: H={H_fit:.2f}, G={G_fit:.2f}")
        ax.invert_yaxis()
        ax.set_xlabel("Solar phase angle  alpha  (deg)")
        ax.set_ylabel(_ylab + ", filter-aligned")
        ax.set_title(f"{label} -- phase curve ({n_pts} night-filter medians, aligned)")
        ax.legend(title="filter")
        fig.tight_layout()
        fig.savefig(PHASE_PNG, dpi=130)
        plt.close(fig)
        display(Image(filename=PHASE_PNG))
    except Exception as _e:
        print(f"phase curve skipped: {type(_e).__name__}: {_e}")



# ==========================================================================
#  CROSS-METHOD AGREEMENT  (ported from model_agreement.ipynb) -- combined CSV
# ==========================================================================
def _ag_fmatch(a, b, tol=AGREE_TOL):
    """True if periods a,b agree to within tol in FREQUENCY (1/P). 1% by default."""
    fa, fb = 1.0 / a, 1.0 / b
    return abs(fa - fb) <= tol * 0.5 * (fa + fb)


def _ag_any_match(P, targets):
    return any(_ag_fmatch(P, t) for t in targets)


def _ag_dedup_pdm(pdm):
    """Drop PDM periods that are a {1,2,1/2,3/2,2/3} multiple of a kept (better-ranked) one."""
    kept = []
    for P in pdm:
        if any(_ag_fmatch(P, k * r) for k in kept for r in AGREE_DEDUP_RATIOS):
            continue
        kept.append(P)
    return kept


def _ag_adjust_pdm(P, ls, fourier):
    """Rescale one PDM period by a single {2,1/2,2/3,3/2} to line up with LS then
    Fourier (PDM harmonics: half / 2x / 1.5x). Returns (adjusted_period, scale)."""
    if _ag_any_match(P, ls) or _ag_any_match(P, fourier):
        return P, 1.0
    for s in AGREE_PDM_SCALES:
        if _ag_any_match(P * s, ls):
            return P * s, s
    for s in AGREE_PDM_SCALES:
        if _ag_any_match(P * s, fourier):
            return P * s, s
    return P, 1.0


def _ag_cluster(items):
    """Union-find over items; two items join if item[0] (period) match by _ag_fmatch.
    Returns a list of clusters, each a list of the original items."""
    n = len(items); parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    for i in range(n):
        for j in range(i + 1, n):
            if _ag_fmatch(items[i][0], items[j][0]):
                union(i, j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(items[i])
    return list(groups.values())


def _obs_per_filter_gate(lc):
    """'yes' if >= OBS_MIN_FILTERS filters each hold >= OBS_MIN_PER_FILTER clipped points."""
    from collections import Counter
    cnt = Counter(np.asarray(lc["band"]).tolist())
    n_ok = sum(1 for v in cnt.values() if v >= OBS_MIN_PER_FILTER)
    return "yes" if n_ok >= OBS_MIN_FILTERS else "no"


def _mbls_grid(baseline_days):
    """Freq grid for the inline FAP: same RANGE as the LS search (P_MIN_HOURS..baseline/2,
    capped 72 h) but oversampled FAP_GRID_OVERSAMPLE (not V4's ~100x). Real + shuffled
    periodograms share this grid, so the FAP is internally valid."""
    P_max_h = min(max(1.0, baseline_days * HOURS_PER_DAY / 2.0), 72)
    f_min = 1.0 / (P_max_h / HOURS_PER_DAY)
    f_max = 1.0 / (P_MIN_HOURS / HOURS_PER_DAY)
    n = max(2, int(round(FAP_GRID_OVERSAMPLE * (f_max - f_min) * baseline_days)))
    return np.linspace(f_min, f_max, n)


def _object_fap(lc, cands, seed_key):
    """Inline permutation FAP (BIC) for an object's gate-passing period(s), from its
    clipped light curve. cands = [(match_rank, raw_P_h), ...] -> {match_rank: FAP}.
    Null: within each filter, shuffle the (mag,err) PAIRS among that filter's times,
    re-run MBLS, take the max power; repeat FAP_N_RANDOM times. FAP = (1 + #{max_rand
    >= real_peak_power}) / (N+1). One null distribution per object, reused for all its
    candidate periods. Empty cands -> {}."""
    if not cands:
        return {}
    from astropy.timeseries import LombScargleMultiband
    import hashlib
    t = np.asarray(lc["t"], float); mag = np.asarray(lc["mag"], float); err = np.asarray(lc["err"], float)
    bands = np.asarray([str(b) for b in lc["band"]])
    freq = _mbls_grid(float(t.max() - t.min()))
    lsm = LombScargleMultiband(t, mag, bands, dy=err, nterms_base=FAP_NTERMS_BASE, nterms_band=FAP_NTERMS_BAND)
    real_pow = np.asarray(lsm.power(freq))
    _win = max(2, int(round(2 * FAP_GRID_OVERSAMPLE)))     # grid-local window around a candidate's raw freq
    real_stats = []
    for rank, raw_P_h in cands:
        f_raw = HOURS_PER_DAY / float(raw_P_h)
        i0 = int(np.argmin(np.abs(freq - f_raw)))
        real_stats.append((int(rank), float(real_pow[max(0, i0 - _win):i0 + _win + 1].max())))
    _seed = FAP_SEED + int.from_bytes(hashlib.sha1(str(seed_key).encode()).digest()[:4], "little")
    rng = np.random.default_rng(_seed)                     # deterministic per object (stable across processes)
    groups = [np.where(bands == b)[0] for b in np.unique(bands)]
    maxpow = np.empty(FAP_N_RANDOM)
    for _k in range(FAP_N_RANDOM):
        pm = mag.copy(); pe = err.copy()
        for g in groups:
            p = rng.permutation(g)          # same permutation for mag & err -> pairs preserved
            pm[g] = mag[p]; pe[g] = err[p]
        lsm_r = LombScargleMultiband(t, pm, bands, dy=pe, nterms_base=FAP_NTERMS_BASE, nterms_band=FAP_NTERMS_BAND)
        maxpow[_k] = float(np.asarray(lsm_r.power(freq)).max())
    return {rank: round((1 + int(np.count_nonzero(maxpow >= stat))) / (FAP_N_RANDOM + 1), 5)
            for rank, stat in real_stats}


_COLOR_COLS = ["u-r", "g-r", "i-r", "z-r", "y-r", "g-i", "r-i"]


def _mbls_offset_colors(lsm, ubands, band, t, mag, matched_period_h):
    """Per-band colours from the MBLS fit at the matched period. astropy centres the
    per-band offsets out of lsm.model, so each band's offset is recovered as its mean
    magnitude AFTER the MBLS shared rotation shape is removed:  offset_b = mean(mag_b -
    shape(t_b))  -- rotation-corrected (unlike a raw median). r is the reference; only
    bands with >= MBLS_COLOR_MIN_OBS obs are used. Blank where a band is absent/sparse
    or r itself is unavailable."""
    cols = {c: "" for c in _COLOR_COLS}
    if matched_period_h is None or matched_period_h <= 0:
        return cols
    freq = HOURS_PER_DAY / float(matched_period_h)                    # cycles/day
    shape = np.asarray(lsm_model_shape(lsm, np.asarray(t, dtype=float), freq, list(ubands)))
    band = np.asarray(band); mag = np.asarray(mag, dtype=float)
    off = {}
    for b in ubands:
        sel = band == b
        if int(sel.sum()) >= MBLS_COLOR_MIN_OBS:
            off[b] = float(np.mean(mag[sel] - shape[sel]))            # rotation-removed band level
    if "r" not in off:
        return cols                                                  # no reference band -> no colours
    _r = off["r"]
    for b in ("u", "g", "i", "z", "y"):
        if b in off:
            cols["%s-r" % b] = round(off[b] - _r, 4)
    if "g" in off and "i" in off:  cols["g-i"] = round(off["g"] - off["i"], 4)
    if "i" in off:                 cols["r-i"] = round(_r - off["i"], 4)
    return cols


def build_agreement_rows(summary, lc):
    """One combined-CSV row per period where ALL THREE methods agree (within 1% in
    frequency; PDM allowed one harmonic rescale). Ranked by agreement strength =
    sum of each method's own rank (LS ranks by power, Fourier by sigma, PDM by
    theta), tie-break by LS power. If nothing matches: a single 'none' row (only the
    obs/filter gate filled). Gates are yes/no, evaluated AT the matched period."""
    def _num(x):
        try:
            v = float(x); return v if (v > 0 and v == v) else None
        except (TypeError, ValueError):
            return None

    def _num0(x):                       # like _num but accepts 0 (a 0 uncertainty is valid, not missing)
        try:
            v = float(x); return v if (v >= 0 and v == v) else None
        except (TypeError, ValueError):
            return None

    ls  = [(P, r, _num(summary.get(f"ls2_P{r}_power_raw")))
           for r in (1, 2, 3) if (P := _num(summary.get(f"ls2_P{r}"))) is not None]
    fo  = [(P, r, _num(summary.get(f"fourier_cand{r}_sigma")))
           for r in (1, 2, 3) if (P := _num(summary.get(f"fourier_cand{r}_P"))) is not None]
    pdm = [(P, r, _num(summary.get(f"pdm_P{r}_theta")))
           for r in (1, 2, 3) if (P := _num(summary.get(f"pdm_P{r}"))) is not None]

    ls_P = [x[0] for x in ls]; fo_P = [x[0] for x in fo]
    pdm_meta = {x[0]: (x[1], x[2]) for x in pdm}
    pdm_adj = []
    for Pp in _ag_dedup_pdm([x[0] for x in pdm]):
        Padj, _s = _ag_adjust_pdm(Pp, ls_P, fo_P)
        rk, th = pdm_meta[Pp]
        pdm_adj.append((Padj, "PDM", rk, th))

    items = ([(P, "LS", rk, pw) for (P, rk, pw) in ls]
           + [(P, "F",  rk, sg) for (P, rk, sg) in fo]
           + pdm_adj)

    matches = []
    for grp in _ag_cluster(items):
        by = {}
        for it in grp:
            by.setdefault(it[1], []).append(it)
        if not ({"LS", "F", "PDM"} <= set(by)):
            continue
        lsm  = min(by["LS"],  key=lambda it: it[2])       # best-ranked member per method
        fom  = min(by["F"],   key=lambda it: it[2])
        pdmm = min(by["PDM"], key=lambda it: it[2])
        rep = (lsm[0] + fom[0] + pdmm[0]) / 3.0
        matches.append({"rep": rep, "rank_sum": lsm[2] + fom[2] + pdmm[2],
                        "ls_rank": lsm[2], "f_rank": fom[2], "pdm_rank_n": pdmm[2], "ls_P": lsm[0],
                        "ls_pow": lsm[3], "f_sig": fom[3], "pdm_th": pdmm[3]})
    matches.sort(key=lambda d: (d["rank_sum"], -(d["ls_pow"] if d["ls_pow"] is not None else 0.0)))

    base = {k: summary.get(k, "") for k in
            ("number", "name", "filters", "n_points", "n_nights", "baseline_days")}
    obs_gate = _obs_per_filter_gate(lc)
    from collections import Counter as _Counter
    _obs_cnt = _Counter([str(b) for b in lc["band"]])
    _obs_str = ";".join("%s:%d" % (b, _obs_cnt[b]) for b in sorted(_obs_cnt))   # e.g. "g:36;i:26;r:59"

    # object-level noise + amplitude error (do NOT depend on which period matched):
    #   median error bar (1-sigma) and  sigma_dm ~= (2*sqrt(2)/sqrt(N)) * sigma_phot
    _err = np.asarray(lc["err"], dtype=float)
    _sig_phot = float(np.median(_err)) if _err.size else float("nan")
    _Nobs = int(len(lc["t"]))
    _amp_err = (2.0 * np.sqrt(2.0) / np.sqrt(_Nobs)) * _sig_phot if _Nobs > 0 else float("nan")
    _med_col = round(_sig_phot, 5) if np.isfinite(_sig_phot) else ""
    _ae_col = round(float(_amp_err), 6) if np.isfinite(_amp_err) else ""

    if not matches:
        row = dict(base)
        row.update({"match_rank": 0, "matched_period": "none",
                    "phase_coverage_pct": "", "gate_phase_coverage": "",
                    "ls_power": "", "gate_ls_power": "",
                    "fourier_sigma": "", "gate_fourier_sigma": "",
                    "pdm_theta": "", "gate_pdm_theta": "",
                    "obs_per_filter": _obs_str, "gate_obs_per_filter": obs_gate,
                    "period_unc": "", "gate_period_unc": "",
                    "mbls_amplitude": "", "median_magerr": _med_col,
                    "amp_error": _ae_col, "amp_snr": "", "gate_amp_snr": "",
                    "FAP": "", "gate_fap": "", "gate_overall": "no",
                    "mbls_rank": "", "pdm_rank": "", "fourier_rank": "",
                    **{_c: "" for _c in _COLOR_COLS}})
        return [row]

    _t = np.asarray(lc["t"], dtype=float); _t0 = float(np.min(_t))
    # MBLS re-fit for per-band offset colours (evaluated at each matched period below)
    from astropy.timeseries import LombScargleMultiband as _LSM
    _cbands = np.asarray([str(_b) for _b in lc["band"]])
    _ccounts = {_b: int((_cbands == _b).sum()) for _b in set(_cbands.tolist())}
    _cub = sorted(set(_cbands.tolist()))
    _clsm = _LSM(_t, np.asarray(lc["mag"], dtype=float), _cbands,
                 dy=np.asarray(lc["err"], dtype=float), nterms_base=2, nterms_band=0)
    _GATE7 = ("gate_phase_coverage", "gate_ls_power", "gate_fourier_sigma", "gate_pdm_theta",
              "gate_obs_per_filter", "gate_period_unc", "gate_amp_snr")
    rows, _passing = [], []
    for i, d in enumerate(matches, start=1):
        _P_match = float(d["ls_P"])                       # matched period = the mbls (multiband-LS) period
        cov_ok, _cov_n = _phase_coverage_ok(np.mod((_t - _t0) * (HOURS_PER_DAY / _P_match), 1.0))
        _pct = round(100.0 * _cov_n / PHASE_COV_NBINS, 1)   # % of the 20 bins with >= 2 points
        # multiband-LS (mbls) period / amplitude / uncertainty for THIS match
        _P_ls = float(d["ls_P"])
        _amp_ls = _num(summary.get(f"ls2_P{d['ls_rank']}_amp"))
        _unc_ls = _num0(summary.get(f"ls2_P{d['ls_rank']}_unc"))
        # period-uncertainty gate (frequency-space): the period uncertainty must be within
        # PERIOD_UNC_FREQ_FRAC of the period -- freq_unc <= FRAC*freq  <=>  period_unc <= FRAC*P.
        if _P_ls > 0:
            _freq = 1.0 / _P_ls
            _punc_thr = max(PERIOD_UNC_GATE_FLOOR_H, (PERIOD_UNC_FREQ_FRAC * _freq) / _freq ** 2)
        else:
            _punc_thr = None
        _gate_punc = "yes" if (_unc_ls is not None and _punc_thr is not None and _unc_ls <= _punc_thr) else "no"
        # amplitude SNR gate: mbls amplitude / amplitude-error >= AMP_SNR_GATE
        _snr = (_amp_ls / _amp_err) if (_amp_ls is not None and np.isfinite(_amp_err) and _amp_err > 0) else None
        _gate_snr = "yes" if (_snr is not None and _snr >= AMP_SNR_GATE) else "no"
        row = dict(base)
        row.update({
            "match_rank": i,
            "matched_period": round(_P_match, 5),
            "phase_coverage_pct": _pct,
            "gate_phase_coverage": "yes" if cov_ok else "no",
            "ls_power": round(d["ls_pow"], 4) if d["ls_pow"] is not None else "",
            "gate_ls_power": "yes" if (d["ls_pow"] is not None and d["ls_pow"] >= LS_POWER_MIN) else "no",
            "fourier_sigma": round(d["f_sig"], 6) if d["f_sig"] is not None else "",
            "gate_fourier_sigma": "yes" if (d["f_sig"] is not None and d["f_sig"] < FOURIER_SIGMA_MAX) else "no",
            "pdm_theta": round(d["pdm_th"], 4) if d["pdm_th"] is not None else "",
            "gate_pdm_theta": "yes" if (d["pdm_th"] is not None and d["pdm_th"] < PDM_THETA_MAX) else "no",
            "obs_per_filter": _obs_str,
            "gate_obs_per_filter": obs_gate,
            "period_unc": round(_unc_ls, 5) if _unc_ls is not None else "",
            "gate_period_unc": _gate_punc,
            "mbls_amplitude": round(_amp_ls, 4) if _amp_ls is not None else "",
            "median_magerr": _med_col,
            "amp_error": _ae_col,
            "amp_snr": round(_snr, 2) if _snr is not None else "",
            "gate_amp_snr": _gate_snr,
            "mbls_rank": "p%d" % int(d["ls_rank"]),        # which LS period (p1/p2/p3) this matched
            "pdm_rank": "p%d" % int(d["pdm_rank_n"]),      # which PDM period
            "fourier_rank": "p%d" % int(d["f_rank"]),      # which Fourier candidate
        })
        row.update(_mbls_offset_colors(_clsm, _cub, _cbands, _t, lc["mag"], _P_match))   # MBLS offset colours
        rows.append(row)
        if all(row[_g] == "yes" for _g in _GATE7):        # passed all 7 gates -> eligible for the BIC/FAP
            _rawP = _num(summary.get(f"ls2_P{d['ls_rank']}_raw"))
            if _rawP is not None:
                _passing.append((len(rows) - 1, i, _rawP))   # (row index, match_rank, raw period h)

    # --- inline BIC / permutation-FAP for the gate-passing period(s) (one null per object) ---
    for _row in rows:
        _row["FAP"] = ""; _row["gate_fap"] = ""; _row["gate_overall"] = "no"
    if _passing:
        _faps = _object_fap(lc, [(_r, _rp) for (_ri, _r, _rp) in _passing], summary.get("name", ""))
        for (_ri, _rank, _rp) in _passing:
            _f = _faps.get(_rank)
            if _f is not None:
                rows[_ri]["FAP"] = _f
                _ok = _f <= FAP_PASS_MAX
                rows[_ri]["gate_fap"] = "yes" if _ok else "no"
                rows[_ri]["gate_overall"] = "yes" if _ok else "no"   # already passed the 7 gates
    return rows

---

# Part 4 &mdash; LSST batch (full SBN X05): reduce, gates, BIC & SBDB

Process **every staged object** (`obs_sbn_X05_full.csv` split into per-object files
with &ge;&nbsp;`MIN_POINTS` points), in parallel across CPU cores. Per object:
HORIZONS geometry at observatory **X05** &rarr; reduce to `H_reduced` &rarr; run all
three period methods &rarr; where all three agree (within 1%, PDM harmonics allowed)
apply the 7 quality **gates** + inline **BIC / permutation FAP** &rarr; enrich from
**JPL SBDB** (orbit class, NEO/PHA, known rotation). Rows append to
`DP1_master_summary.csv` and matched periods to `DP1_combined_agreement.csv`
(crash-safe, resumable; transient HORIZONS failures are retried at the end).

In [21]:
from datetime import datetime

FULL_CSV  = os.path.join(BASE_DIR, "obs_sbn_X05_full.csv")   # 1.7 GB Rubin/LSST obs (obs_sbn X05)
RAW_DIR   = os.path.join(BASE_DIR, "_raw_by_object")         # per-object staged raw photometry
INDEX_CSV = os.path.join(BASE_DIR, "object_index.csv")       # object -> n_points (from the split)
MPC_CODE  = "X05"                                            # observatory (Rubin); HORIZONS site + night binning


def obstime_to_jd(s):
    """ISO UTC timestamp ('2025-05-04T03:08:23.328Z') -> Julian Date (UTC).

    Pure arithmetic (Fliegel-Van Flandern) so it is fast enough for ~10^7 rows;
    matches the old 'mjd + 2400000.5' convention (UTC JD, no leap-second shift)."""
    s = s.strip()
    if s.endswith("Z"):
        s = s[:-1]
    dt = datetime.fromisoformat(s)                       # naive, treated as UTC
    a = (14 - dt.month) // 12
    y = dt.year + 4800 - a
    m = dt.month + 12 * a - 3
    jdn = dt.day + (153 * m + 2) // 5 + 365 * y + y // 4 - y // 100 + y // 400 - 32045
    frac = (dt.hour - 12) / 24 + dt.minute / 1440 + (dt.second + dt.microsecond / 1e6) / 86400
    return jdn + frac


def _obj_slug(obj):
    s = str(obj).strip()
    for ch in r' \/:*?"<>|':                              # filesystem-illegal chars (e.g. comet "C/2024 F2")
        s = s.replace(ch, "_")
    return s


def _raw_path(obj):
    return os.path.join(RAW_DIR, _obj_slug(obj) + ".csv")


import re

_COMET_NUM = re.compile(r'^\d+[PD](?:[/\-]|$)')          # numbered periodic comet: 1P, 289P, 73P-C


def is_comet(name):
    """True for comet designations: C/ P/ D/ X/ prefixes, or numbered NNNP / NNND.
    Kept as asteroids: numeric ids, 'YYYY LLn' provisionals, and A/ asteroidal objects."""
    n = str(name).strip()
    if _COMET_NUM.match(n):
        return True
    return "/" in n and n.split("/", 1)[0] in ("C", "P", "D", "X")


def build_object_files(min_points=MIN_POINTS, flush_every=2_000_000):
    """ONE streaming pass over FULL_CSV -> per-object raw CSVs (jd,mag,magerr,filter,
    session_date) in RAW_DIR, plus INDEX_CSV of point counts.

    Object key: permid if present, else provid ('easier one' when both).  Rows with
    neither id, or an unparseable time/mag, are ignored.  Objects with < min_points
    total points are pruned afterwards (they'd be skipped downstream anyway)."""
    import shutil
    if os.path.isdir(RAW_DIR):                              # start clean so a re-run never appends to a partial build
        shutil.rmtree(RAW_DIR)
    os.makedirs(RAW_DIR, exist_ok=True)
    buf, counts, started = {}, {}, set()
    n_rows = n_noid = n_comet = n_bad = 0
    buffered = 0

    def flush():
        nonlocal buffered
        for obj, lines in buf.items():
            new = obj not in started
            with open(_raw_path(obj), "a", encoding="utf-8", newline="") as fh:
                if new:
                    fh.write("jd,mag,magerr,filter,session_date\n"); started.add(obj)
                fh.writelines(lines)
        buf.clear(); buffered = 0

    with open(FULL_CSV, encoding="utf-8", errors="replace", newline="") as fh:
        rdr = csv.reader(fh)
        header = next(rdr)
        ix = {n: header.index(n) for n in
              ("permid", "provid", "obstime_text", "mag", "rmsmag", "band")}
        for row in rdr:
            n_rows += 1
            obj = row[ix["permid"]].strip() or row[ix["provid"]].strip()   # permid preferred
            if not obj:
                n_noid += 1; continue
            if is_comet(obj):                              # asteroids only -> drop comets
                n_comet += 1; continue
            try:
                jd = obstime_to_jd(row[ix["obstime_text"]])
                mag = float(row[ix["mag"]])
            except (ValueError, IndexError):
                n_bad += 1; continue
            try:
                magerr = float(row[ix["rmsmag"]])
            except (ValueError, IndexError):
                magerr = float("nan")
            band = _norm_band(row[ix["band"]])
            date = row[ix["obstime_text"]][:10]
            buf.setdefault(obj, []).append(f"{jd:.8f},{mag:.4f},{magerr},{band},{date}\n")
            counts[obj] = counts.get(obj, 0) + 1
            buffered += 1
            if buffered >= flush_every:
                flush()
                print(f"  ... {n_rows:,} rows scanned, {len(counts):,} objects so far")
    flush()

    kept = 0
    for obj, cnt in counts.items():
        if cnt < min_points:
            p = _raw_path(obj)
            if os.path.exists(p):
                os.remove(p)
        else:
            kept += 1
    with open(INDEX_CSV, "w", encoding="utf-8", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["object", "n_points"])
        for obj in sorted(counts, key=lambda o: -counts[o]):
            w.writerow([obj, counts[obj]])

    print(f"scanned {n_rows:,} rows: {n_noid:,} no-id, {n_comet:,} comet, {n_bad:,} bad time/mag (all ignored)")
    print(f"{len(counts):,} distinct objects; {kept:,} with >= {min_points} points -> {RAW_DIR}")
    print(f"index -> {INDEX_CSV}")
    return counts


def reduce_rubin(obj_id, output_csv, cache_csv):
    """Staged per-object photometry -> HORIZONS geometry -> reduced CSV.
    (Run build_object_files() once first to create the per-object raw files.)"""
    raw = _raw_path(obj_id)
    if not os.path.exists(raw):
        raise RuntimeError(f"{obj_id}: no staged file {os.path.basename(raw)} "
                           f"(run build_object_files() first)")
    recs = []
    with open(raw, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            _me = row["magerr"]
            recs.append({"jd": float(row["jd"]), "mag": float(row["mag"]),
                         "magerr": float(_me) if _me not in ("", "nan") else float("nan"),
                         "filter": _norm_band(row["filter"]),
                         "session_date": row["session_date"], "phase_meta": None})
    if not recs:
        raise RuntimeError(f"{obj_id}: empty staged file")
    if len(recs) < MIN_POINTS:                               # too few points -> skip before any geometry
        raise SkipObject(f"{obj_id}: {len(recs)} raw points < {MIN_POINTS}")
    recs.sort(key=lambda r: r["jd"])
    didx = {d: i for i, d in enumerate(sorted({r["session_date"] for r in recs}))}
    for r in recs:
        r["session"] = didx[r["session_date"]]

    jds = [r["jd"] for r in recs]
    geom = query_horizons(jds, target=obj_id, site=MPC_CODE, cache_path=cache_csv)
    recs = reduce_records(recs, geom)
    write_csv(recs, output_csv)

In [22]:
# --- ONE-TIME: split the 1.7 GB obs_sbn_X05_full.csv into per-object raw files ------
# Multi-minute streaming pass; safe to re-run (skips if the index already exists).
if not os.path.exists(INDEX_CSV):
    build_object_files()
else:
    print(f"{os.path.basename(INDEX_CSV)} already present -- delete it (and _raw_by_object) to rebuild.")

  ... 2,037,490 rows scanned, 95,538 objects so far
scanned 2,972,809 rows: 64,630 no-id, 158 comet, 0 bad time/mag (all ignored)
131,566 distinct objects; 7,055 with >= 50 points -> E:\CLAUDE\_raw_by_object
index -> E:\CLAUDE\object_index.csv


In [ ]:
def analyze_rubin(obj_id):
    """Full pipeline for one Rubin object; returns its one-row summary dict."""
    global P_MIN_HOURS, P_MAX_HOURS
    slug = obj_id.replace(" ", "_")
    label = obj_id
    outdir = os.path.join(BASE_DIR, slug)
    os.makedirs(outdir, exist_ok=True)
    input_csv = os.path.join(outdir, f"{slug}_reduced.csv")
    cache_csv = os.path.join(outdir, f"{slug}_horizons_cache.csv")
    if not os.path.exists(input_csv):
        reduce_rubin(obj_id, input_csv, cache_csv)

    _raw = load_lightcurve(input_csv)
    if len(_raw["t"]) < MIN_POINTS:                          # too few points (cached CSV) -> skip, no row
        raise SkipObject(f"{label}: {len(_raw['t'])} points < {MIN_POINTS}")
    lc, removed, _ = clip_outliers(_raw, CLIP_K)
    print(f"[{label}] kept {len(lc['t'])} pts; flagged {len(removed['t'])} outlier(s).")
    # obs-per-filter pre-filter: need >= OBS_MIN_PER_FILTER obs in >= OBS_MIN_FILTERS filters
    # (clipped light curve) -- else skip the object entirely, no CSV row (like the <MIN_POINTS check).
    from collections import Counter as _Counter
    _oc = _Counter([str(_b) for _b in lc["band"]])
    if sum(1 for _v in _oc.values() if _v >= OBS_MIN_PER_FILTER) < OBS_MIN_FILTERS:
        raise SkipObject(f"{label}: <{OBS_MIN_FILTERS} filters with >={OBS_MIN_PER_FILTER} obs "
                         f"({';'.join('%s:%d' % (_b, _oc[_b]) for _b in sorted(_oc))})")
    P_MIN_HOURS = 0.0156                                     # period window: 0.1 h .. baseline/2
    P_MAX_HOURS = min(max(1.0, float(lc["t"].max() - lc["t"].min()) * HOURS_PER_DAY / 2.0), 72)
    sessions, cmap = session_palette(lc)

    summary = {
        "number": obj_id if str(obj_id).strip().isdigit() else "",   # numbered designation only (blank if provisional)
        "name": obj_id,                                              # the object's designation (always populated)
        "filters": ";".join(sorted(set(lc["band"]))),
        "n_points": int(len(lc["t"])), "n_nights": int(len(set(lc["session"]))),
        "baseline_days": round(float(lc["t"].max() - lc["t"].min()), 4),
        "H_abs_mean": round(float(np.mean(lc["mag"])), 4),
        "H_abs_std": round(float(np.std(lc["mag"])), 4),
        "obs_per_filter": ";".join("%s:%d" % (_b, _oc[_b]) for _b in sorted(_oc)),   # per-filter clipped counts
    }
    plot_outliers(lc, removed, sessions, cmap, slug, label, outdir)
    analyze_multiband(lc, sessions, cmap, slug, label, outdir, summary)
    analyze_pdm(lc, sessions, cmap, slug, label, outdir, summary)
    analyze_fourier(lc, sessions, cmap, slug, label, outdir, summary)
    analyze_ls_residual(lc, sessions, cmap, slug, label, outdir)
    analyze_phasecurve(lc, sessions, cmap, slug, label, outdir, summary)
    summary["comment"] = summary.pop("comment", "")
    summary.update(_sbdb_lookup(obj_id))                 # JPL SBDB class/neo/pha/rotation -> last columns
    agreement_rows = build_agreement_rows(summary, lc)   # matched-period rows for the combined CSV
    return summary, agreement_rows


# inspect ONE object interactively (shows its plots inline):
#   analyze_rubin("2025 MA19")

In [ ]:
# --- Rubin batch (MULTIPROCESSING): run objects in parallel, one per CPU core ---
# Same pipeline/logic as before -- analyze_rubin() and every analysis method are
# UNCHANGED. Only the batch *driver* changed: instead of one object at a time, we
# fan objects out across (CPU cores - 1) worker processes and collect results as
# they finish. The master CSV is still written by THIS (parent) process only, one
# row per completed object, so it stays crash-safe with no concurrent-write races.
#
# Windows note: workers are spawned (fresh interpreters) and cannot see functions
# defined in notebook cells. We ship analyze_rubin to them by *value* with
# cloudpickle (already installed) via a tiny generic on-disk shim (_cprun.py) --
# this is exactly what joblib/loky do internally. No pipeline code lives in the shim.
import pandas as pd, time, io, contextlib, os, sys, cloudpickle
import concurrent.futures as _cf
import multiprocessing as _mp

# Target list = every staged object with >= MIN_POINTS points (from object_index.csv).
_ix = pd.read_csv(INDEX_CSV)
RUBIN_TARGETS = [str(o) for o, n in zip(_ix["object"], _ix["n_points"]) if n >= MIN_POINTS]
PRIORITY_OBJECTS = []       # empty for now -- paste designations here to prioritise


if PRIORITY_OBJECTS:
    _staged = set(RUBIN_TARGETS)
    _pri = [o for o in PRIORITY_OBJECTS if o in _staged]
    RUBIN_TARGETS = _pri + [o for o in RUBIN_TARGETS if o not in set(_pri)]
print(f"{len(RUBIN_TARGETS)} object(s) with >= {MIN_POINTS} points in {os.path.basename(INDEX_CSV)}")

_out = os.path.join(BASE_DIR, "DP1_master_summary.csv")
_out_combined = os.path.join(BASE_DIR, "DP1_combined_agreement.csv")   # matched-period gate rows
_COMBINED_COLS = ["number", "name", "filters", "n_points", "n_nights", "baseline_days",
                  "match_rank", "matched_period",
                  "phase_coverage_pct", "gate_phase_coverage",
                  "ls_power", "gate_ls_power",
                  "fourier_sigma", "gate_fourier_sigma",
                  "pdm_theta", "gate_pdm_theta",
                  "obs_per_filter", "gate_obs_per_filter",
                  "period_unc", "gate_period_unc",
                  "mbls_amplitude", "median_magerr", "amp_error", "amp_snr", "gate_amp_snr",
                  "FAP", "gate_fap", "gate_overall",
                  "mbls_rank", "pdm_rank", "fourier_rank",
                  "u-r", "g-r", "i-r", "z-r", "y-r", "g-i", "r-i"]

# --- resume: if a master already exists, skip objects already in it and append the rest ---
_done, _cols = set(), None
if os.path.exists(_out):
    try:
        _prev = pd.read_csv(_out)
        _cols = list(_prev.columns)                       # append aligned to the existing schema
        _done = set(_prev["name"].astype(str))            # 'name' column holds the obj_id (always populated)
        print(f"resume: {len(_done)} object(s) already in {os.path.basename(_out)} -- skipping those.")
    except Exception as _e:
        print(f"resume: couldn't read existing master ({_e}); starting fresh.")
        _done, _cols = set(), None

_todo = [o for o in RUBIN_TARGETS if o not in _done]
print(f"{len(_todo)} object(s) to process; {len(_done)} already done.")

# ---------------------------------------------------------------------------
# worker-side wrapper (shipped by value to each process). Returns primitives
# ONLY -- (tag, obj, payload, secs) -- so nothing custom has to pickle back.
# ---------------------------------------------------------------------------
def _run_one(_obj):
    _t0 = time.time()
    try:
        with contextlib.redirect_stdout(io.StringIO()):   # silence per-method chatter
            _r = analyze_rubin(_obj)
        return ("ok", _obj, _r, time.time() - _t0)
    except SkipObject as _e:
        return ("skip", _obj, str(_e), time.time() - _t0)
    except Exception as _e:
        return ("fail", _obj, f"{type(_e).__name__}: {str(_e)[:120]}", time.time() - _t0)

# once-per-worker environment setup: matplotlib is imported fresh in each spawned
# worker, so re-apply the SAME headless backend + font sizes as the notebook top
# (cell "find_period.py") -- otherwise worker plots use matplotlib's smaller
# default fonts (text looks smaller than the sequential run).
def _worker_setup():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.size": 16,
        "axes.titlesize": 19,
        "axes.labelsize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 15,
        "legend.title_fontsize": 16,
    })

# tiny generic shim the spawned workers import (no pipeline logic lives here) -----
_SHIM = os.path.join(BASE_DIR, "_cprun.py")
with open(_SHIM, "w", encoding="utf-8") as _fh:
    _fh.write(
        "import cloudpickle\n"
        "_FN = None\n"
        "def init(fn_bytes, setup_bytes=None):\n"
        "    global _FN\n"
        "    if setup_bytes is not None:\n"
        "        cloudpickle.loads(setup_bytes)()   # once-per-worker env setup (matplotlib fonts/backend)\n"
        "    _FN = cloudpickle.loads(fn_bytes)   # rebuild analyze_rubin by value, once per worker\n"
        "def run(args):\n"
        "    return _FN(*args)\n"
    )
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)                          # so spawned workers can import _cprun
import _cprun

# silence inline images during the batch, THEN capture analyze_rubin (with the
# no-op display baked in) for the workers; restore display for interactive use.
_display_off = (lambda *a, **k: None)
_display_saved = display
display = _display_off
try:
    _fn_bytes = cloudpickle.dumps(_run_one)              # serialize pipeline ONCE (reused by all workers)
    _setup_bytes = cloudpickle.dumps(_worker_setup)      # matplotlib backend + font rcParams for each worker
finally:
    display = _display_saved

N_WORKERS = max(1, (os.cpu_count() or 2) - 3)            # leave one core free for other usage
print(f"dispatching across {N_WORKERS} worker process(es)...")

# transient (network / HORIZONS 5xx) failures are RE-QUEUED and retried after the main
# pass, when the API load has dropped -- with fewer workers + a pause so it can recover.
RETRY_ROUNDS  = 3         # extra passes over the still-failing objects
RETRY_PAUSE_S = 60        # seconds to wait before each retry round

def _is_transient(msg):
    m = str(msg).lower()
    return any(k in m for k in ("503", "502", "504", "500 server", "timeout", "timed out",
               "temporarily unavailable", "service unavailable", "connection",
               "httperror", "max retries", "remote end closed"))

_n_ok = 0
_N = len(_todo)
_pending = list(_todo)          # objects still to attempt (transient failures carry forward)
_hard_failed = []               # (obj, msg) failures that were not retried / kept failing
_round = 0
while _pending:
    _is_retry = _round > 0
    _rn_workers = N_WORKERS if not _is_retry else max(1, N_WORKERS // 3)   # ease HORIZONS load on retries
    if _is_retry:
        print("")
        print(f"--- retry round {_round}/{RETRY_ROUNDS}: {len(_pending)} transient failure(s); "
              f"{_rn_workers} worker(s) after {RETRY_PAUSE_S}s pause ---")
        time.sleep(RETRY_PAUSE_S)
    _next, _M, _m_done = [], len(_pending), 0
    with _cf.ProcessPoolExecutor(
            max_workers=_rn_workers,
            mp_context=_mp.get_context("spawn"),
            initializer=_cprun.init, initargs=(_fn_bytes, _setup_bytes),
            max_tasks_per_child=10) as _ex:               # recycle workers to bound memory/mpl state
        _futs = {_ex.submit(_cprun.run, (_obj,)): _obj for _obj in _pending}
        for _fut in _cf.as_completed(_futs):
            _m_done += 1
            _tag, _obj, _payload, _secs = _fut.result()   # never raises: wrapper returns primitives
            _pfx = f"[retry {_round}: {_m_done}/{_M}]" if _is_retry else f"[{_m_done}/{_N}]"
            if _tag == "ok":
                _summary, _agree = _payload                # analyze_rubin -> (summary, agreement_rows)
                # --- append THIS object to the master file right away (crash-safe) ---
                if _cols is None:                          # brand-new file: first row sets the header
                    _cols = list(_summary.keys())
                    pd.DataFrame([_summary], columns=_cols).to_csv(_out, mode="w", header=True, index=False)
                else:                                      # align to existing columns, append one row
                    pd.DataFrame([_summary]).reindex(columns=_cols).to_csv(_out, mode="a", header=False, index=False)
                # --- append matched-period agreement row(s) to the combined CSV ---
                if _agree:
                    pd.DataFrame(_agree).reindex(columns=_COMBINED_COLS).to_csv(
                        _out_combined, mode="a", header=not os.path.exists(_out_combined), index=False)
                _done.add(_obj)
                _n_ok += 1
                _mrow = _agree[0] if _agree else {}
                print(f"{_pfx} {_obj:<11} OK  {_secs:5.1f}s  "
                      f"n={_summary['n_points']}  match_P={_mrow.get('matched_period')}  "
                      f"-> appended ({_n_ok})")
            elif _tag == "skip":
                print(f"{_pfx} {_obj:<11} SKIP  ({_payload})")
            else:
                if _is_transient(_payload) and _round < RETRY_ROUNDS:
                    _next.append(_obj)                     # transient -> retry in a later round
                    print(f"{_pfx} {_obj:<11} transient fail -> will retry  ({_payload[:70]})")
                else:
                    _hard_failed.append((_obj, _payload))  # permanent, or out of retries
                    print(f"{_pfx} {_obj:<11} FAILED: {_payload}")
    _pending = _next
    _round += 1

print("")
print(f"done: {_n_ok} new object(s); {len(_hard_failed)} still failed after {RETRY_ROUNDS} retr(ies); "
      f"master -> {_out}; combined agreement -> {_out_combined}")
if _hard_failed:
    print("  still-failed (re-run this cell to retry -- resume skips the done ones):")
    for _o, _m in _hard_failed[:25]:
        print(f"    {_o}: {_m[:90]}")
